# Study C reproducible 40-run analysis with borehole-disjoint training and test data

This standalone notebook consolidates the NoSpatial, XY, Z, and XYZ workflows for BM2 Study C. It generates mean and sample standard deviation over 40 paired seeds, 95% confidence intervals, constituent-model results, ensemble results, RMSE improvement tables, and paired Wilcoxon run-to-run stability tests with Holm correction.

Important safeguards:

- The supplied training and test files are used directly; no separate validation CSV is combined with the training data.
- Training and test `global_id` values and borehole identifiers must be disjoint. The notebook stops if either overlap is detected.
- Every eligible observation in the 106-record test file is evaluated for each applicable Missing Type and target.
- The test set is not used for hyperparameter tuning or model selection.
- Hyperparameter tuning uses five-fold borehole-grouped cross-validation within the training data.
- Preprocessing is fitted inside each cross-validation fold.
- Spatial features are assigned explicitly for every Missing Type.
- The pre-specified ensemble is the arithmetic mean of RF, GBR, and XGBoost predictions.


## Requirements and input files

Place `BM2_Data_All_without_106_records.csv` and `BM2_Data_All_with_106_records.csv` in the same directory as this notebook, or enter their full paths in the settings cell. The former is used as the Study C training/development dataset and the latter as the borehole-disjoint test dataset. Both files must contain `global_id`, `borehole_id`, the index and mechanical variables, and the `x`, `y`, and `z_depth` coordinates required by the workflow. A `rows` column is optional because Study C evaluates all test observations because Study C evaluates all supplied test observations directly.

Required packages are `numpy`, `pandas`, `scipy`, `scikit-learn`, `xgboost`, `joblib`, `matplotlib`, and `openpyxl`. The optional `psutil` package adds physical-core and memory information to the system table. The spatial-comparator sections use NumPy, SciPy, and scikit-learn and require no additional geostatistical package.


## Imports, constants, and experiment schema

In [1]:
from __future__ import annotations

import argparse
import json
import platform
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import joblib
import numpy as np
import pandas as pd
import scipy
from scipy.stats import t as student_t
from scipy.stats import wilcoxon
import sklearn
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


CORE_FEATURES = [
    "Sr",
    "gamma_t",
    "e",
    "LL",
    "PL",
    "w",
    "PI",
    "LI",
    "e_squared",
    "w_by_LL",
    "Sr_log",
    "moisture_void_ratio",
]

SPATIAL_FEATURES = {
    "NoSpatial": [],
    "XY": ["x", "y"],
    "Z": ["z_depth"],
    "XYZ": ["x", "y", "z_depth"],
}

CONFIG_LABELS = {
    "NoSpatial": "Without Spatial",
    "XY": "With XY",
    "Z": "With Z",
    "XYZ": "With XYZ",
}

TARGET_ORDER = ["Cc", "Cv", "Pc", "E50", "su"]
CONFIG_ORDER = ["NoSpatial", "XY", "Z", "XYZ"]
MISSING_TYPE_ORDER = ["MT1", "MT2", "MT3", "MT4"]
MODEL_ORDER = ["RF", "GBR", "XGB", "Ensemble"]
SPLIT_ORDER = ["Development", "Test"]

MISSINGNESS_PATTERNS = {
    "MT1": {
        "known_mechanical": [],
        "targets": ["Cc", "Cv", "Pc", "E50", "su"],
        "polynomial_degree": 1,
    },
    "MT2": {
        "known_mechanical": ["su"],
        "targets": ["Cc", "Cv", "Pc", "E50"],
        "polynomial_degree": 2,
    },
    "MT3": {
        "known_mechanical": ["su", "E50"],
        "targets": ["Cc", "Cv", "Pc"],
        "polynomial_degree": 2,
    },
    "MT4": {
        "known_mechanical": ["Pc", "Cc", "Cv"],
        "targets": ["E50", "su"],
        "polynomial_degree": 2,
    },
}


@dataclass(frozen=True)
class ExperimentTask:
    configuration: str
    missing_type: str
    target: str
    features: tuple[str, ...]
    polynomial_degree: int


## Definitions and experiment schema

In [2]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Generate reproducible Study C mean +/- SD tables from 40 seeded runs."
    )
    parser.add_argument(
        "--train",
        default="BM2_Data_All_without_106_records.csv",
        help="Study C training/development CSV.",
    )
    parser.add_argument(
        "--test",
        default="BM2_Data_All_with_106_records.csv",
        help="Study C borehole-disjoint test CSV; all eligible rows are evaluated.",
    )
    parser.add_argument("--output-dir", default="StudyC_40run_results")
    parser.add_argument("--runs", type=int, default=40)
    parser.add_argument("--base-seed", type=int, default=2026)
    parser.add_argument("--tuning-seed", type=int, default=1729)
    parser.add_argument("--cv-folds", type=int, default=5)
    parser.add_argument(
        "--cv-strategy",
        choices=["group", "random"],
        default="group",
        help="Use borehole-grouped CV by default; choose random to mirror the notebooks.",
    )
    parser.add_argument(
        "--search-profile",
        choices=["expanded", "manuscript", "smoke"],
        default="expanded",
        help="Expanded addresses reviewer concerns; manuscript mirrors Table 1; smoke is for testing.",
    )
    parser.add_argument(
        "--search-iterations",
        type=int,
        default=20,
        help="Randomized-search iterations per model for the expanded profile.",
    )
    parser.add_argument(
        "--tuning-score",
        choices=["rmse", "r2"],
        default="rmse",
        help="Hyperparameter selection objective. RMSE is aligned with the manuscript tables.",
    )
    parser.add_argument("--n-jobs", type=int, default=-1)
    parser.add_argument("--huber-delta", type=float, default=1.0)
    parser.add_argument(
        "--configurations",
        nargs="+",
        choices=CONFIG_ORDER,
        default=CONFIG_ORDER,
    )
    parser.add_argument(
        "--missing-types",
        nargs="+",
        choices=MISSING_TYPE_ORDER,
        default=MISSING_TYPE_ORDER,
    )
    parser.add_argument(
        "--targets",
        nargs="+",
        choices=TARGET_ORDER,
        default=None,
        help="Optional target subset for diagnostics or smoke runs.",
    )
    parser.add_argument(
        "--models",
        nargs="+",
        choices=["RF", "GBR", "XGB"],
        default=["RF", "GBR", "XGB"],
    )
    parser.add_argument(
        "--save-models",
        action="store_true",
        help="Save every fitted constituent model. This can require substantial disk space.",
    )
    parser.add_argument(
        "--validate-only",
        action="store_true",
        help="Validate inputs and write manifests without fitting models.",
    )
    return parser.parse_args()


## Arguments, input loading, and leakage checks

In [3]:
def resolve_input_path(raw_path: str) -> Path:
    supplied = Path(raw_path).expanduser()
    script_location = globals().get("__file__")
    script_dir = Path(script_location).resolve().parent if script_location else Path.cwd()
    candidates = [supplied]
    if not supplied.is_absolute():
        candidates.extend(
            [
                Path.cwd() / supplied,
                script_dir / supplied,
                script_dir / "upload" / supplied.name,
            ]
        )
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not find {raw_path!r}. Supply an explicit path with --train or --test."
    )


def load_and_engineer(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame = frame.loc[:, ~frame.columns.str.startswith("Unnamed:")].copy()
    required_raw = {
        "global_id",
        "borehole_id",
        "Sr",
        "gamma_t",
        "e",
        "LL",
        "PL",
        "w",
        "su",
        "E50",
        "Pc",
        "Cc",
        "Cv",
        "x",
        "y",
        "z_depth",
    }
    missing = sorted(required_raw.difference(frame.columns))
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    frame["PI"] = frame["LL"] - frame["PL"]
    frame["LI"] = (frame["w"] - frame["PL"]) / (frame["PI"] + 1e-5)
    frame["e_squared"] = frame["e"] ** 2
    frame["w_by_LL"] = frame["w"] / (frame["LL"] + 1e-5)
    if (frame["Sr"] + 1e-5 <= 0).any():
        raise ValueError(f"{path.name} contains Sr values that cannot be log transformed.")
    frame["Sr_log"] = np.log(frame["Sr"] + 1e-5)
    frame["moisture_void_ratio"] = frame["w"] * frame["e"]
    frame.replace([np.inf, -np.inf], np.nan, inplace=True)

    check_columns = sorted(
        set(CORE_FEATURES)
        | {"x", "y", "z_depth", "su", "E50", "Pc", "Cc", "Cv"}
    )
    missing_counts = frame[check_columns].isna().sum()
    missing_counts = missing_counts[missing_counts > 0]
    if not missing_counts.empty:
        raise ValueError(
            f"{path.name} has missing or infinite modeling values: {missing_counts.to_dict()}"
        )
    return frame


def select_test_rows(test: pd.DataFrame, missing_type: str) -> pd.DataFrame:
    """Return all Study C test observations for an applicable Missing Type."""
    if missing_type not in MISSINGNESS_PATTERNS:
        raise KeyError(f"Unknown Missing Type: {missing_type!r}")
    return test.reset_index(drop=True).copy()


def prepare_splits(
    train: pd.DataFrame,
    test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[dict[str, Any]]]:
    """Validate and label the supplied borehole-disjoint Study C datasets."""
    for split_name, frame in [("training", train), ("test", test)]:
        duplicated_ids = frame.loc[
            frame["global_id"].duplicated(keep=False), "global_id"
        ].tolist()
        if duplicated_ids:
            raise ValueError(
                f"The {split_name} file contains duplicated global_id values: "
                f"{sorted(set(duplicated_ids))[:20]}"
            )

    train_ids = set(train["global_id"])
    test_ids = set(test["global_id"])
    record_overlap = train_ids & test_ids
    if record_overlap:
        raise ValueError(
            "Test leakage detected: global_id values occur in both training and test data: "
            f"{sorted(record_overlap)[:20]}"
        )

    train_boreholes = set(train["borehole_id"])
    test_boreholes = set(test["borehole_id"])
    borehole_overlap = train_boreholes & test_boreholes
    if borehole_overlap:
        raise ValueError(
            "Borehole leakage detected: borehole identifiers occur in both training and "
            f"test data: {sorted(borehole_overlap, key=str)[:20]}"
        )

    development = train.reset_index(drop=True).copy()
    development["source_split"] = "train"
    test = test.reset_index(drop=True).copy()
    test["source_split"] = "test"

    audit = [
        {"item": "training_rows_supplied", "value": len(development)},
        {"item": "test_rows_supplied", "value": len(test)},
        {"item": "training_test_overlapping_ids", "value": len(record_overlap)},
        {"item": "training_test_overlapping_boreholes", "value": len(borehole_overlap)},
        {"item": "development_unique_rows_used", "value": development["global_id"].nunique()},
        {"item": "test_unique_rows_used", "value": test["global_id"].nunique()},
        {"item": "development_boreholes", "value": development["borehole_id"].nunique()},
        {"item": "test_boreholes", "value": test["borehole_id"].nunique()},
    ]
    for missing_type in MISSING_TYPE_ORDER:
        audit.append(
            {"item": f"{missing_type}_test_record_count", "value": len(test)}
        )
    return development, test, audit


def iter_tasks(args: argparse.Namespace) -> Iterable[ExperimentTask]:
    requested_targets = set(args.targets) if args.targets else None
    for missing_type in MISSING_TYPE_ORDER:
        if missing_type not in args.missing_types:
            continue
        pattern = MISSINGNESS_PATTERNS[missing_type]
        for configuration in CONFIG_ORDER:
            if configuration not in args.configurations:
                continue
            features = tuple(
                CORE_FEATURES
                + SPATIAL_FEATURES[configuration]
                + list(pattern["known_mechanical"])
            )
            for target in pattern["targets"]:
                if requested_targets is not None and target not in requested_targets:
                    continue
                yield ExperimentTask(
                    configuration=configuration,
                    missing_type=missing_type,
                    target=target,
                    features=features,
                    polynomial_degree=int(pattern["polynomial_degree"]),
                )


## Model construction and hyperparameter search

In [4]:
def get_xgb_regressor(seed: int) -> Any:
    try:
        import xgboost
        from xgboost import XGBRegressor
    except ImportError as exc:
        raise ImportError(
            "XGBoost is required for the three-model manuscript ensemble. "
            "Install it with `pip install xgboost`, or use --models RF GBR only for a smoke test."
        ) from exc
    return XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=seed,
        n_jobs=1,
        verbosity=0,
    )


def base_model(model_name: str, seed: int) -> Any:
    if model_name == "RF":
        return RandomForestRegressor(random_state=seed, n_jobs=1)
    if model_name == "GBR":
        return GradientBoostingRegressor(random_state=seed, loss="squared_error")
    if model_name == "XGB":
        return get_xgb_regressor(seed)
    raise KeyError(model_name)


def search_space(model_name: str, profile: str) -> dict[str, list[Any]]:
    prefix = "regressor__model__"
    if profile == "smoke":
        spaces = {
            "RF": {"n_estimators": [20], "max_depth": [5], "min_samples_split": [2]},
            "GBR": {"n_estimators": [20], "learning_rate": [0.05], "max_depth": [3]},
            "XGB": {"n_estimators": [20], "learning_rate": [0.05], "max_depth": [3]},
        }
    elif profile == "manuscript":
        spaces = {
            "RF": {
                "n_estimators": [100, 200, 500],
                "max_depth": [5, 10],
                "min_samples_split": [2, 5],
            },
            "GBR": {
                "n_estimators": [100, 200, 500],
                "learning_rate": [0.05, 0.1],
                "max_depth": [3, 5],
            },
            "XGB": {
                "n_estimators": [100, 200, 500],
                "learning_rate": [0.05, 0.1],
                "max_depth": [3, 5],
            },
        }
    else:
        spaces = {
            "RF": {
                "n_estimators": [200, 500, 800],
                "max_depth": [None, 5, 10, 20],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4],
                "max_features": [1.0, 0.5, "sqrt"],
            },
            "GBR": {
                "n_estimators": [100, 200, 500],
                "learning_rate": [0.03, 0.05, 0.1],
                "max_depth": [2, 3, 5],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4],
                "subsample": [0.8, 1.0],
                "max_features": [None, "sqrt"],
            },
            "XGB": {
                "n_estimators": [100, 200, 500],
                "learning_rate": [0.03, 0.05, 0.1],
                "max_depth": [3, 5, 8],
                "min_child_weight": [1, 3, 5],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0],
                "reg_alpha": [0.0, 0.1],
                "reg_lambda": [1.0, 5.0],
            },
        }
    return {prefix + key: value for key, value in spaces[model_name].items()}


def build_estimator(model_name: str, degree: int, seed: int) -> TransformedTargetRegressor:
    pipeline = Pipeline(
        [
            ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
            ("scale", StandardScaler()),
            ("model", base_model(model_name, seed)),
        ]
    )
    return TransformedTargetRegressor(regressor=pipeline, transformer=StandardScaler())


def make_cv(
    strategy: str,
    folds: int,
    seed: int,
    groups: pd.Series,
) -> tuple[Any, np.ndarray | None]:
    if strategy == "random":
        return KFold(n_splits=folds, shuffle=True, random_state=seed), None
    unique_groups = groups.nunique()
    if unique_groups < folds:
        raise ValueError(f"Grouped CV requested {folds} folds but only {unique_groups} groups exist.")
    try:
        splitter = GroupKFold(n_splits=folds, shuffle=True, random_state=seed)
    except TypeError:
        splitter = GroupKFold(n_splits=folds)
    return splitter, groups.to_numpy()


def tune_task_models(
    task: ExperimentTask,
    development: pd.DataFrame,
    args: argparse.Namespace,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    X = development.loc[:, task.features]
    y = development.loc[:, task.target]
    cv, groups = make_cv(
        args.cv_strategy,
        args.cv_folds,
        args.tuning_seed,
        development["borehole_id"],
    )
    scoring = "neg_root_mean_squared_error" if args.tuning_score == "rmse" else "r2"
    estimators: dict[str, Any] = {}
    records: list[dict[str, Any]] = []

    for model_name in args.models:
        estimator = build_estimator(model_name, task.polynomial_degree, args.tuning_seed)
        params = search_space(model_name, args.search_profile)
        if args.search_profile == "expanded":
            search = RandomizedSearchCV(
                estimator=estimator,
                param_distributions=params,
                n_iter=args.search_iterations,
                scoring=scoring,
                cv=cv,
                random_state=args.tuning_seed,
                n_jobs=args.n_jobs,
                refit=True,
                return_train_score=False,
            )
        else:
            search = GridSearchCV(
                estimator=estimator,
                param_grid=params,
                scoring=scoring,
                cv=cv,
                n_jobs=args.n_jobs,
                refit=True,
                return_train_score=False,
            )
        fit_kwargs = {"groups": groups} if groups is not None else {}
        started = time.perf_counter()
        search.fit(X, y, **fit_kwargs)
        elapsed = time.perf_counter() - started
        estimators[model_name] = clone(search.best_estimator_)
        records.append(
            {
                "configuration": task.configuration,
                "configuration_label": CONFIG_LABELS[task.configuration],
                "missing_type": task.missing_type,
                "target": task.target,
                "model": model_name,
                "features": ", ".join(task.features),
                "raw_feature_count": len(task.features),
                "polynomial_degree": task.polynomial_degree,
                "expanded_feature_count": _expanded_feature_count(
                    len(task.features), task.polynomial_degree
                ),
                "cv_strategy": args.cv_strategy,
                "cv_folds": args.cv_folds,
                "tuning_score": args.tuning_score,
                "search_profile": args.search_profile,
                "best_cv_score": float(search.best_score_),
                "best_parameters": json.dumps(search.best_params_, sort_keys=True),
                "tuning_seconds": elapsed,
            }
        )
    return estimators, records


def _expanded_feature_count(raw_count: int, degree: int) -> int:
    if degree == 1:
        return raw_count
    if degree == 2:
        return raw_count + raw_count * (raw_count + 1) // 2
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    poly.fit(np.zeros((1, raw_count)))
    return int(poly.n_output_features_)


def reseed_estimator(estimator: Any, seed: int, n_jobs: int) -> Any:
    fitted = clone(estimator)
    available = fitted.get_params(deep=True)
    updates: dict[str, Any] = {}
    for name in available:
        if name.endswith("random_state"):
            updates[name] = seed
        elif name.endswith("n_jobs"):
            updates[name] = n_jobs
    if updates:
        fitted.set_params(**updates)
    return fitted


## Repeated-run evaluation and metrics

Development metrics use the complete Study C training set. Test predictions and metrics use every observation in the supplied borehole-disjoint test file for each applicable Missing Type and target.


In [5]:
def robust_l2(y_true: np.ndarray, y_pred: np.ndarray, delta: float) -> float:
    residual = np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float)
    absolute = np.abs(residual)
    loss = np.where(
        absolute <= delta,
        0.5 * residual**2,
        delta * (absolute - 0.5 * delta),
    )
    return float(np.mean(loss))


def calculate_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    huber_delta: float,
) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    l2 = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(l2))
    denominator = float(np.mean(np.abs(y_true)))
    rmse_percent = np.nan if denominator == 0 else 100.0 * rmse / denominator
    return {
        "rmse": rmse,
        "rmse_percent": rmse_percent,
        "l2": l2,
        "robust_l2": robust_l2(y_true, y_pred, huber_delta),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def evaluate_task(
    task: ExperimentTask,
    tuned_estimators: dict[str, Any],
    development: pd.DataFrame,
    test: pd.DataFrame,
    args: argparse.Namespace,
    output_dir: Path,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    task_test = select_test_rows(test, task.missing_type)
    X_development = development.loc[:, task.features]
    y_development = development.loc[:, task.target].to_numpy()
    X_test = task_test.loc[:, task.features]
    y_test = task_test.loc[:, task.target].to_numpy()
    metric_rows: list[dict[str, Any]] = []
    prediction_rows: list[dict[str, Any]] = []

    for run_index in range(args.runs):
        seed = args.base_seed + run_index
        predictions: dict[str, dict[str, np.ndarray]] = {
            "Development": {},
            "Test": {},
        }
        for model_name, template in tuned_estimators.items():
            estimator = reseed_estimator(template, seed, args.n_jobs)
            started = time.perf_counter()
            estimator.fit(X_development, y_development)
            fit_seconds = time.perf_counter() - started
            predictions["Development"][model_name] = estimator.predict(X_development)
            predictions["Test"][model_name] = estimator.predict(X_test)
            if args.save_models:
                model_dir = output_dir / "models" / task.configuration / task.missing_type / task.target
                model_dir.mkdir(parents=True, exist_ok=True)
                joblib.dump(estimator, model_dir / f"{model_name}_seed_{seed}.joblib")

            for split, truth in [("Development", y_development), ("Test", y_test)]:
                values = calculate_metrics(
                    truth,
                    predictions[split][model_name],
                    args.huber_delta,
                )
                metric_rows.append(
                    _metric_record(task, run_index, seed, model_name, split, len(truth), fit_seconds, values)
                )

        for split, truth in [("Development", y_development), ("Test", y_test)]:
            ensemble_prediction = np.mean(
                np.column_stack([predictions[split][name] for name in args.models]),
                axis=1,
            )
            predictions[split]["Ensemble"] = ensemble_prediction
            values = calculate_metrics(truth, ensemble_prediction, args.huber_delta)
            metric_rows.append(
                _metric_record(task, run_index, seed, "Ensemble", split, len(truth), np.nan, values)
            )

        for row_position, (_, source_row) in enumerate(task_test.iterrows()):
            record = {
                "run": run_index + 1,
                "seed": seed,
                "configuration": task.configuration,
                "configuration_label": CONFIG_LABELS[task.configuration],
                "missing_type": task.missing_type,
                "target": task.target,
                "test_row": (
                    int(source_row["rows"])
                    if "rows" in source_row.index
                    else int(position)
                ),
                "global_id": source_row["global_id"],
                "borehole_id": source_row["borehole_id"],
                "actual": y_test[row_position],
            }
            for model_name in args.models + ["Ensemble"]:
                record[f"prediction_{model_name}"] = predictions["Test"][model_name][row_position]
            prediction_rows.append(record)

        print(
            f"  completed {task.configuration} {task.missing_type} {task.target}, "
            f"run {run_index + 1}/{args.runs}, seed={seed}",
            flush=True,
        )
    return metric_rows, prediction_rows


def _metric_record(
    task: ExperimentTask,
    run_index: int,
    seed: int,
    model: str,
    split: str,
    sample_count: int,
    fit_seconds: float,
    values: dict[str, float],
) -> dict[str, Any]:
    return {
        "run": run_index + 1,
        "seed": seed,
        "configuration": task.configuration,
        "configuration_label": CONFIG_LABELS[task.configuration],
        "missing_type": task.missing_type,
        "target": task.target,
        "model": model,
        "split": split,
        "n": sample_count,
        "fit_seconds": fit_seconds,
        **values,
    }


## Uncertainty summaries and paired statistical tests

In [6]:
def summarize_metrics(run_metrics: pd.DataFrame) -> pd.DataFrame:
    identifiers = [
        "configuration",
        "configuration_label",
        "missing_type",
        "target",
        "model",
        "split",
        "n",
    ]
    metrics = ["rmse", "rmse_percent", "l2", "robust_l2", "mae", "r2"]
    grouped = run_metrics.groupby(identifiers, sort=False, dropna=False)
    rows: list[dict[str, Any]] = []
    for keys, group in grouped:
        row = dict(zip(identifiers, keys))
        row["runs"] = len(group)
        critical = student_t.ppf(0.975, df=max(len(group) - 1, 1))
        for metric in metrics:
            values = group[metric].to_numpy(dtype=float)
            mean = float(np.mean(values))
            std = float(np.std(values, ddof=1)) if len(values) > 1 else np.nan
            margin = critical * std / np.sqrt(len(values)) if len(values) > 1 else np.nan
            row[f"{metric}_mean"] = mean
            row[f"{metric}_sd"] = std
            row[f"{metric}_ci95_low"] = mean - margin if len(values) > 1 else np.nan
            row[f"{metric}_ci95_high"] = mean + margin if len(values) > 1 else np.nan
        rows.append(row)
    result = pd.DataFrame(rows)
    return sort_results(result)


def calculate_improvements(run_metrics: pd.DataFrame) -> pd.DataFrame:
    ensemble = run_metrics.loc[run_metrics["model"] == "Ensemble"].copy()
    baseline = ensemble.loc[ensemble["configuration"] == "NoSpatial", [
        "run", "seed", "missing_type", "target", "split", "rmse", "rmse_percent"
    ]].rename(columns={"rmse": "baseline_rmse", "rmse_percent": "baseline_rmse_percent"})
    spatial = ensemble.loc[ensemble["configuration"] != "NoSpatial"].copy()
    merged = spatial.merge(
        baseline,
        on=["run", "seed", "missing_type", "target", "split"],
        how="left",
        validate="many_to_one",
    )
    if merged[["baseline_rmse", "baseline_rmse_percent"]].isna().any().any():
        raise ValueError("Improvement tables require NoSpatial results for every selected task.")
    merged["rmse_improvement_percent"] = 100.0 * (
        merged["baseline_rmse"] - merged["rmse"]
    ) / merged["baseline_rmse"]
    merged["rmse_percent_improvement_percent"] = 100.0 * (
        merged["baseline_rmse_percent"] - merged["rmse_percent"]
    ) / merged["baseline_rmse_percent"]
    columns = [
        "run",
        "seed",
        "configuration",
        "configuration_label",
        "missing_type",
        "target",
        "split",
        "baseline_rmse",
        "rmse",
        "rmse_improvement_percent",
        "baseline_rmse_percent",
        "rmse_percent",
        "rmse_percent_improvement_percent",
    ]
    return sort_results(merged[columns])


def summarize_improvements(improvements: pd.DataFrame) -> pd.DataFrame:
    identifiers = ["configuration", "configuration_label", "missing_type", "target", "split"]
    rows: list[dict[str, Any]] = []
    for keys, group in improvements.groupby(identifiers, sort=False):
        row = dict(zip(identifiers, keys))
        row["runs"] = len(group)
        for metric in ["rmse_improvement_percent", "rmse_percent_improvement_percent"]:
            values = group[metric].to_numpy(float)
            row[f"{metric}_mean"] = float(values.mean())
            row[f"{metric}_sd"] = float(values.std(ddof=1)) if len(values) > 1 else np.nan
        rows.append(row)
    return sort_results(pd.DataFrame(rows))


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted_sorted = np.empty_like(p_values)
    running = 0.0
    total = len(p_values)
    for rank, original_index in enumerate(order):
        candidate = min(1.0, (total - rank) * p_values[original_index])
        running = max(running, candidate)
        adjusted_sorted[original_index] = running
    return adjusted_sorted


def paired_tests(run_metrics: pd.DataFrame) -> pd.DataFrame:
    ensemble_test = run_metrics.loc[
        (run_metrics["model"] == "Ensemble") & (run_metrics["split"] == "Test")
    ].copy()
    rows: list[dict[str, Any]] = []
    for (missing_type, target, configuration), spatial in ensemble_test.loc[
        ensemble_test["configuration"] != "NoSpatial"
    ].groupby(["missing_type", "target", "configuration"], sort=False):
        baseline = ensemble_test.loc[
            (ensemble_test["missing_type"] == missing_type)
            & (ensemble_test["target"] == target)
            & (ensemble_test["configuration"] == "NoSpatial"),
            ["seed", "rmse"],
        ].rename(columns={"rmse": "baseline_rmse"})
        paired = spatial[["seed", "rmse"]].merge(baseline, on="seed", validate="one_to_one")
        difference = paired["rmse"].to_numpy() - paired["baseline_rmse"].to_numpy()
        if len(difference) < 2 or np.allclose(difference, 0):
            statistic, p_value = np.nan, 1.0
        else:
            result = wilcoxon(difference, alternative="two-sided", zero_method="wilcox")
            statistic, p_value = float(result.statistic), float(result.pvalue)
        sd = float(np.std(difference, ddof=1)) if len(difference) > 1 else np.nan
        rows.append(
            {
                "missing_type": missing_type,
                "target": target,
                "configuration": configuration,
                "configuration_label": CONFIG_LABELS[configuration],
                "paired_runs": len(difference),
                "mean_rmse_difference_spatial_minus_baseline": float(np.mean(difference)),
                "median_rmse_difference_spatial_minus_baseline": float(np.median(difference)),
                "paired_effect_size_dz": float(np.mean(difference) / sd) if sd and np.isfinite(sd) else np.nan,
                "wilcoxon_statistic": statistic,
                "p_value_two_sided": p_value,
            }
        )
    result = pd.DataFrame(rows)
    if not result.empty:
        result["p_value_holm"] = holm_adjust(result["p_value_two_sided"].to_numpy())
        result["significant_holm_0.05"] = result["p_value_holm"] < 0.05
    return sort_results(result)


## Manuscript tables and exports

In [7]:
def mean_sd_text(mean: float, sd: float, target: str, metric: str) -> str:
    if pd.isna(mean):
        return ""
    if target == "Cc" and metric not in {"rmse_percent", "r2"}:
        digits = 4
    elif metric in {"rmse_percent", "r2"}:
        digits = 3
    elif target in {"E50", "Cv"}:
        digits = 2
    else:
        digits = 3
    if pd.isna(sd):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f} +/- {sd:.{digits}f}"


def manuscript_performance_table(summary: pd.DataFrame, missing_type: str) -> pd.DataFrame:
    data = summary.loc[
        (summary["model"] == "Ensemble") & (summary["missing_type"] == missing_type)
    ].copy()
    rows: list[dict[str, Any]] = []
    metrics = ["rmse", "rmse_percent", "l2", "robust_l2"]
    for configuration in CONFIG_ORDER:
        for target in TARGET_ORDER:
            subset = data.loc[
                (data["configuration"] == configuration) & (data["target"] == target)
            ]
            if subset.empty:
                continue
            row: dict[str, Any] = {
                "Configuration": CONFIG_LABELS[configuration],
                "Target": target,
            }
            for split in SPLIT_ORDER:
                split_row = subset.loc[subset["split"] == split]
                if split_row.empty:
                    continue
                values = split_row.iloc[0]
                for metric in metrics:
                    label = {
                        "rmse": "RMSE",
                        "rmse_percent": "RMSE%",
                        "l2": "L2",
                        "robust_l2": "Robust L2",
                    }[metric]
                    row[f"{split} {label}"] = mean_sd_text(
                        values[f"{metric}_mean"], values[f"{metric}_sd"], target, metric
                    )
            rows.append(row)
    return pd.DataFrame(rows)


def manuscript_ablation_table(summary: pd.DataFrame) -> pd.DataFrame:
    data = summary.loc[summary["split"] == "Test"].copy()
    rows: list[dict[str, Any]] = []
    for missing_type in MISSING_TYPE_ORDER:
        for target in TARGET_ORDER:
            for configuration in CONFIG_ORDER:
                subset = data.loc[
                    (data["missing_type"] == missing_type)
                    & (data["target"] == target)
                    & (data["configuration"] == configuration)
                ]
                if subset.empty:
                    continue
                row = {
                    "Missing Type": missing_type,
                    "Target": target,
                    "Configuration": CONFIG_LABELS[configuration],
                }
                for model in MODEL_ORDER:
                    model_row = subset.loc[subset["model"] == model]
                    if model_row.empty:
                        continue
                    values = model_row.iloc[0]
                    row[f"{model} test RMSE"] = mean_sd_text(
                        values["rmse_mean"], values["rmse_sd"], target, "rmse"
                    )
                rows.append(row)
    return pd.DataFrame(rows)


def manuscript_improvement_table(improvement_summary: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for missing_type in MISSING_TYPE_ORDER:
        for target in TARGET_ORDER:
            for configuration in ["XY", "Z", "XYZ"]:
                subset = improvement_summary.loc[
                    (improvement_summary["missing_type"] == missing_type)
                    & (improvement_summary["target"] == target)
                    & (improvement_summary["configuration"] == configuration)
                ]
                if subset.empty:
                    continue
                row = {
                    "Missing Type": missing_type,
                    "Target": target,
                    "Configuration": CONFIG_LABELS[configuration],
                }
                for split in SPLIT_ORDER:
                    split_row = subset.loc[subset["split"] == split]
                    if split_row.empty:
                        continue
                    values = split_row.iloc[0]
                    mean = values["rmse_improvement_percent_mean"]
                    sd = values["rmse_improvement_percent_sd"]
                    row[f"{split} RMSE improvement (%)"] = (
                        f"{mean:.2f} +/- {sd:.2f}" if pd.notna(sd) else f"{mean:.2f}"
                    )
                rows.append(row)
    return pd.DataFrame(rows)


def sort_results(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    result = frame.copy()
    sort_columns: list[str] = []
    temporary: list[str] = []
    mappings = {
        "configuration": CONFIG_ORDER,
        "missing_type": MISSING_TYPE_ORDER,
        "target": TARGET_ORDER,
        "model": MODEL_ORDER,
        "split": SPLIT_ORDER,
    }
    for column, order in mappings.items():
        if column in result.columns:
            helper = f"__order_{column}"
            result[helper] = pd.Categorical(result[column], categories=order, ordered=True)
            sort_columns.append(helper)
            temporary.append(helper)
    for column in ["run", "seed", "test_row"]:
        if column in result.columns:
            sort_columns.append(column)
    if sort_columns:
        result = result.sort_values(sort_columns, kind="stable")
    return result.drop(columns=temporary).reset_index(drop=True)


def write_excel(output_path: Path, sheets: dict[str, pd.DataFrame]) -> None:
    try:
        with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
            for sheet_name, frame in sheets.items():
                safe_name = sheet_name[:31]
                frame.to_excel(writer, sheet_name=safe_name, index=False)
                worksheet = writer.book[safe_name]
                worksheet.freeze_panes = "A2"
                worksheet.auto_filter.ref = worksheet.dimensions
                for column_cells in worksheet.columns:
                    width = min(
                        55,
                        max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
                        + 2,
                    )
                    worksheet.column_dimensions[column_cells[0].column_letter].width = width
    except ImportError as exc:
        warnings.warn(f"Excel workbook was not written because openpyxl is unavailable: {exc}")


def software_versions() -> dict[str, str]:
    versions = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__,
        "joblib": joblib.__version__,
    }
    try:
        import xgboost

        versions["xgboost"] = xgboost.__version__
    except ImportError:
        versions["xgboost"] = "not installed"
    return versions


def write_manifests(
    output_dir: Path,
    args: argparse.Namespace,
    tasks: list[ExperimentTask],
    audit: list[dict[str, Any]],
    input_paths: dict[str, Path],
) -> None:
    pd.DataFrame(audit).to_csv(output_dir / "sample_count_and_overlap_audit.csv", index=False)
    feature_rows = []
    for task in tasks:
        feature_rows.append(
            {
                **asdict(task),
                "configuration_label": CONFIG_LABELS[task.configuration],
                "features": ", ".join(task.features),
                "raw_feature_count": len(task.features),
                "expanded_feature_count": _expanded_feature_count(
                    len(task.features), task.polynomial_degree
                ),
            }
        )
    pd.DataFrame(feature_rows).to_csv(output_dir / "feature_manifest.csv", index=False)
    manifest = {
        "arguments": vars(args),
        "input_paths": {key: str(value) for key, value in input_paths.items()},
        "software_versions": software_versions(),
        "metric_definitions": {
            "rmse": "sqrt(mean((actual - predicted)^2))",
            "rmse_percent": "100 * RMSE / mean(abs(actual))",
            "l2": "mean((actual - predicted)^2)",
            "robust_l2": (
                "mean Huber loss: 0.5*r^2 when abs(r)<=delta; "
                "delta*(abs(r)-0.5*delta) otherwise"
            ),
            "huber_delta": args.huber_delta,
            "standard_deviation": "sample SD across paired random seeds (ddof=1)",
        },
        "selection_rule": (
            "Hyperparameters selected using development data only. Test labels are used only "
            "for locked final evaluation. The ensemble is a pre-specified arithmetic mean."
        ),
    }
    (output_dir / "experiment_manifest.json").write_text(
        json.dumps(manifest, indent=2, default=str), encoding="utf-8"
    )


## Experiment driver

In [8]:
def main() -> int:
    pipeline_started = time.perf_counter()
    args = parse_args()
    if args.runs < 2 and not args.validate_only:
        warnings.warn("At least two runs are required to estimate a standard deviation.")
    if args.huber_delta <= 0:
        raise ValueError("--huber-delta must be positive.")
    if args.search_iterations < 1:
        raise ValueError("--search-iterations must be at least 1.")

    paths = {
        "train": resolve_input_path(args.train),
        "test": resolve_input_path(args.test),
    }
    output_dir = Path(args.output_dir).expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    print("Loading and validating Study C data...", flush=True)
    supplied_train = load_and_engineer(paths["train"])
    supplied_test = load_and_engineer(paths["test"])
    development, test, audit = prepare_splits(supplied_train, supplied_test)
    tasks = list(iter_tasks(args))
    if not tasks:
        raise ValueError("No experiment tasks remain after applying the requested filters.")
    write_manifests(output_dir, args, tasks, audit, paths)

    print(pd.DataFrame(audit).to_string(index=False), flush=True)
    print(f"Experiment tasks: {len(tasks)}", flush=True)
    if args.validate_only:
        pipeline_seconds = time.perf_counter() - pipeline_started
        pd.DataFrame(
            [{
                "stage": "Input validation and manifest generation",
                "elapsed_seconds": pipeline_seconds,
                "elapsed_minutes": pipeline_seconds / 60.0,
            }]
        ).to_csv(output_dir / "pipeline_wall_clock_time.csv", index=False)
        print(f"Validation complete. Manifests written to {output_dir}")
        return 0
    if "XGB" in args.models:
        # Fail before any expensive tuning if the required package is absent.
        get_xgb_regressor(args.tuning_seed)
    if "NoSpatial" not in args.configurations:
        warnings.warn("NoSpatial was omitted, so improvement and paired-test tables cannot be generated.")

    all_metrics: list[dict[str, Any]] = []
    all_predictions: list[dict[str, Any]] = []
    all_hyperparameters: list[dict[str, Any]] = []

    for task_number, task in enumerate(tasks, start=1):
        print(
            f"[{task_number}/{len(tasks)}] Tuning {task.configuration} "
            f"{task.missing_type} {task.target} using development data only...",
            flush=True,
        )
        tuned, tuning_records = tune_task_models(task, development, args)
        all_hyperparameters.extend(tuning_records)
        metric_rows, prediction_rows = evaluate_task(
            task, tuned, development, test, args, output_dir
        )
        all_metrics.extend(metric_rows)
        all_predictions.extend(prediction_rows)

        pd.DataFrame(all_hyperparameters).to_csv(
            output_dir / "selected_hyperparameters_checkpoint.csv", index=False
        )
        pd.DataFrame(all_metrics).to_csv(
            output_dir / "run_metrics_checkpoint.csv", index=False
        )

    run_metrics = sort_results(pd.DataFrame(all_metrics))
    predictions = sort_results(pd.DataFrame(all_predictions))
    hyperparameters = sort_results(pd.DataFrame(all_hyperparameters))
    summary = summarize_metrics(run_metrics)

    run_metrics.to_csv(output_dir / "run_metrics_all_models.csv", index=False)
    predictions.to_csv(output_dir / "test_predictions_all_runs.csv", index=False)
    hyperparameters.to_csv(output_dir / "selected_hyperparameters.csv", index=False)
    summary.to_csv(output_dir / "summary_metrics_mean_sd_ci95.csv", index=False)

    table2 = manuscript_performance_table(summary, "MT1")
    table3 = manuscript_ablation_table(summary)
    table2.to_csv(output_dir / "Table2_StudyC_MT1_mean_SD.csv", index=False)
    table3.to_csv(output_dir / "Table3_StudyC_model_ablation_mean_SD.csv", index=False)

    per_missing_type: dict[str, pd.DataFrame] = {}
    for missing_type in MISSING_TYPE_ORDER:
        table = manuscript_performance_table(summary, missing_type)
        if table.empty:
            continue
        per_missing_type[missing_type] = table
        table.to_csv(
            output_dir / f"StudyC_{missing_type}_performance_mean_SD.csv", index=False
        )

    improvement_runs = pd.DataFrame()
    improvement_summary = pd.DataFrame()
    tests = pd.DataFrame()
    table4 = pd.DataFrame()
    if "NoSpatial" in args.configurations and any(
        config in args.configurations for config in ["XY", "Z", "XYZ"]
    ):
        improvement_runs = calculate_improvements(run_metrics)
        improvement_summary = summarize_improvements(improvement_runs)
        tests = paired_tests(run_metrics)
        table4 = manuscript_improvement_table(improvement_summary)
        improvement_runs.to_csv(output_dir / "improvements_by_run.csv", index=False)
        improvement_summary.to_csv(output_dir / "improvements_mean_SD.csv", index=False)
        tests.to_csv(output_dir / "paired_wilcoxon_tests_test_RMSE.csv", index=False)
        table4.to_csv(
            output_dir / "Table4_StudyC_train_test_improvements_mean_SD.csv", index=False
        )

    sheets: dict[str, pd.DataFrame] = {
        "Table2_MT1": table2,
        "Table3_Ablation": table3,
        "Metrics_Summary": summary,
        "Hyperparameters": hyperparameters,
        "Sample_Audit": pd.DataFrame(audit),
    }
    for missing_type, table in per_missing_type.items():
        sheets[f"Performance_{missing_type}"] = table
    if not table4.empty:
        sheets["Table4_Improvements"] = table4
        sheets["Improvement_Numeric"] = improvement_summary
        sheets["Paired_Tests"] = tests
    write_excel(output_dir / "StudyC_manuscript_tables_40_runs.xlsx", sheets)

    pipeline_seconds = time.perf_counter() - pipeline_started
    pd.DataFrame(
        [{
            "stage": "Complete Study C experiment",
            "elapsed_seconds": pipeline_seconds,
            "elapsed_minutes": pipeline_seconds / 60.0,
        }]
    ).to_csv(output_dir / "pipeline_wall_clock_time.csv", index=False)
    print(
        f"Completed in {pipeline_seconds / 60.0:.2f} minutes. "
        f"Results written to {output_dir}",
        flush=True,
    )
    return 0


## Experiment settings

In [9]:
# Edit this cell only if the Study C files are stored elsewhere.
NOTEBOOK_ARGS = [
    "--train", "BM2_Train_borehole_grouped_80.csv",
    "--test", "BM2_Test_borehole_grouped_20.csv",
    "--output-dir", "StudyB_40run_results_Oct1",
    "--runs", "40",
    "--base-seed", "2026",
    "--tuning-seed", "1729",
    "--cv-folds", "5",
    "--cv-strategy", "group",
    "--search-profile", "expanded",
    "--search-iterations", "20",
    "--tuning-score", "rmse",
    "--n-jobs", "-1",
    "--huber-delta", "1.0",
]

# Set to True only when ready for the complete experiment.
# The expanded 56-task analysis performs thousands of model fits.
RUN_FULL_EXPERIMENT = False


## Notebook execution helper

In [10]:
def run_pipeline(notebook_args):
    """Run the same argument-driven pipeline safely inside Jupyter."""
    original_argv = sys.argv.copy()
    try:
        sys.argv = ["StudyC_40run_reproducible_tables"] + list(notebook_args)
        return main()
    finally:
        sys.argv = original_argv


## Validate inputs before fitting

This validation also confirms that the test file contains every required `rows` value exactly once.


In [11]:
# Fast schema, record-overlap, borehole-overlap, and feature-manifest validation.
run_pipeline(NOTEBOOK_ARGS + ["--validate-only"])


Loading and validating Study C data...
                               item  value
             training_rows_supplied   2219
                 test_rows_supplied    555
      training_test_overlapping_ids      0
training_test_overlapping_boreholes      0
       development_unique_rows_used   2219
              test_unique_rows_used    555
              development_boreholes    184
                     test_boreholes     52
              MT1_test_record_count    555
              MT2_test_record_count    555
              MT3_test_record_count    555
              MT4_test_record_count    555
Experiment tasks: 56
Validation complete. Manifests written to /scratch/pm4167/BM2/BM2_Revision_Sept/StudyB/StudyB_40run_results_Oct1


0

## Run the complete experiment

The default expanded search directly addresses the reviewer request for broader and equal hyperparameter optimization. For a closer reproduction of the submitted Table 1 grid, change `expanded` to `manuscript`.


In [12]:
if RUN_FULL_EXPERIMENT:
    run_pipeline(NOTEBOOK_ARGS)
else:
    print(
        "Validation is complete. Set RUN_FULL_EXPERIMENT = True in the settings cell "
        "and rerun this cell to launch the complete 40-run analysis."
    )


Validation is complete. Set RUN_FULL_EXPERIMENT = True in the settings cell and rerun this cell to launch the complete 40-run analysis.


## Borehole-disjoint training and validation loss curves across runs

These curves are diagnostics, not a second model-selection step. For each seed, the development pool is split by borehole, preprocessing is fitted on the training portion only, and the locked hyperparameters selected by the main pipeline are reused. XGBoost uses its per-iteration evaluation history, Gradient Boosting uses staged predictions, and Random Forest uses warm-started tree checkpoints. RMSE is calculated on the target standardized from the training subset.


In [13]:
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit

LOSS_CURVE_MODEL_ORDER = ["XGB", "GBR", "RF"]
LOSS_CURVE_MODEL_LABELS = {
    "XGB": "XGBoost", "GBR": "Gradient Boosting", "RF": "Random Forest",
}
LOSS_CURVE_TARGET_LABELS = {
    "su": r"$S_u$", "E50": r"$E_{50}$", "Pc": r"$P_c$",
    "Cc": r"$C_c$", "Cv": r"$C_v$",
}


def load_development_pool(args: argparse.Namespace) -> pd.DataFrame:
    """Load the supplied Study C training set without adding validation records."""
    return load_and_engineer(resolve_input_path(args.train)).reset_index(drop=True)


def selected_model_parameters(
    selected: pd.DataFrame,
    missing_type: str,
    configuration: str,
    target: str,
    model_name: str,
) -> dict[str, Any]:
    match = selected.loc[
        (selected["missing_type"] == missing_type)
        & (selected["configuration"] == configuration)
        & (selected["target"] == target)
        & (selected["model"] == model_name)
    ]
    if len(match) != 1:
        raise ValueError(
            "Expected one selected-hyperparameter row for "
            f"{missing_type}/{configuration}/{target}/{model_name}, found {len(match)}."
        )
    raw = json.loads(match.iloc[0]["best_parameters"])
    prefix = "regressor__model__"
    return {
        (key[len(prefix):] if key.startswith(prefix) else key): value
        for key, value in raw.items()
    }


def fit_curve_preprocessing(
    train_frame: pd.DataFrame,
    validation_frame: pd.DataFrame,
    features: list[str],
    target: str,
    polynomial_degree: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    polynomial = PolynomialFeatures(degree=polynomial_degree, include_bias=False)
    x_train = polynomial.fit_transform(train_frame.loc[:, features])
    x_validation = polynomial.transform(validation_frame.loc[:, features])
    x_scaler = StandardScaler()
    x_train = x_scaler.fit_transform(x_train)
    x_validation = x_scaler.transform(x_validation)
    y_scaler = StandardScaler()
    y_train = y_scaler.fit_transform(
        train_frame.loc[:, [target]].to_numpy(dtype=float)
    ).ravel()
    y_validation = y_scaler.transform(
        validation_frame.loc[:, [target]].to_numpy(dtype=float)
    ).ravel()
    return x_train, x_validation, y_train, y_validation


def rmse_value(actual: np.ndarray, predicted: np.ndarray) -> float:
    return float(np.sqrt(mean_squared_error(actual, predicted)))


def one_model_loss_curve(
    model_name: str,
    parameters: dict[str, Any],
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_validation: np.ndarray,
    y_validation: np.ndarray,
    seed: int,
    rf_max_points: int,
) -> pd.DataFrame:
    params = dict(parameters)
    params.pop("random_state", None)
    params.pop("n_jobs", None)

    if model_name == "XGB":
        model = get_xgb_regressor(seed)
        model.set_params(n_jobs=1, **params)
        model.fit(
            x_train, y_train,
            eval_set=[(x_train, y_train), (x_validation, y_validation)],
            verbose=False,
        )
        history = model.evals_result()
        train_values = history["validation_0"]["rmse"]
        validation_values = history["validation_1"]["rmse"]
        iterations = np.arange(1, len(train_values) + 1)
    elif model_name == "GBR":
        model = GradientBoostingRegressor(
            random_state=seed, loss="squared_error", **params
        )
        model.fit(x_train, y_train)
        train_values = [
            rmse_value(y_train, prediction)
            for prediction in model.staged_predict(x_train)
        ]
        validation_values = [
            rmse_value(y_validation, prediction)
            for prediction in model.staged_predict(x_validation)
        ]
        iterations = np.arange(1, len(train_values) + 1)
    elif model_name == "RF":
        total_trees = int(params.pop("n_estimators", 100))
        checkpoints = np.unique(
            np.linspace(1, total_trees, min(total_trees, rf_max_points), dtype=int)
        )
        model = RandomForestRegressor(
            random_state=seed, n_jobs=1, warm_start=True, n_estimators=1, **params
        )
        train_values, validation_values = [], []
        for tree_count in checkpoints:
            model.set_params(n_estimators=int(tree_count))
            model.fit(x_train, y_train)
            train_values.append(rmse_value(y_train, model.predict(x_train)))
            validation_values.append(
                rmse_value(y_validation, model.predict(x_validation))
            )
        iterations = checkpoints
    else:
        raise KeyError(model_name)

    return pd.DataFrame({
        "iteration": iterations,
        "train_rmse": np.asarray(train_values, dtype=float),
        "validation_rmse": np.asarray(validation_values, dtype=float),
    })


In [14]:
from importlib import metadata as importlib_metadata
import os
import subprocess

try:
    from IPython.display import display
except ImportError:
    def display(value: Any) -> None:
        print(value)


def notebook_args_namespace(notebook_args: list[str]) -> argparse.Namespace:
    """Parse NOTEBOOK_ARGS without permanently changing the Jupyter kernel arguments."""
    original_argv = sys.argv[:]
    try:
        sys.argv = ["study_c_notebook"] + list(notebook_args)
        return parse_args()
    finally:
        sys.argv = original_argv


def installed_version(distribution: str) -> str:
    try:
        return importlib_metadata.version(distribution)
    except importlib_metadata.PackageNotFoundError:
        return "not installed"


def cpu_model_name() -> str:
    processor = platform.processor().strip()
    if processor:
        return processor
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.is_file():
        for line in cpuinfo.read_text(encoding="utf-8", errors="ignore").splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[-1].strip()
    return "not reported"


def gpu_description() -> str:
    try:
        completed = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,driver_version",
                "--format=csv,noheader",
            ],
            check=True,
            capture_output=True,
            text=True,
            timeout=10,
        )
        values = [line.strip() for line in completed.stdout.splitlines() if line.strip()]
        return "; ".join(values) if values else "not detected"
    except (FileNotFoundError, subprocess.SubprocessError):
        return "not detected"


def system_configuration_table() -> pd.DataFrame:
    try:
        import psutil

        physical_cores = psutil.cpu_count(logical=False)
        logical_cores = psutil.cpu_count(logical=True)
        memory_gib = psutil.virtual_memory().total / (1024 ** 3)
        memory_text = f"{memory_gib:.2f} GiB"
    except ImportError:
        physical_cores = "not reported"
        logical_cores = os.cpu_count() or "not reported"
        memory_text = "not reported (install psutil)"

    rows = [
        ("Operating system", platform.platform()),
        ("Machine architecture", platform.machine() or "not reported"),
        ("CPU model", cpu_model_name()),
        ("Physical CPU cores", physical_cores),
        ("Logical CPU cores", logical_cores),
        ("System memory", memory_text),
        ("GPU name, memory, driver", gpu_description()),
        ("Python implementation", platform.python_implementation()),
        ("Python version", platform.python_version()),
    ]
    return pd.DataFrame(rows, columns=["Component", "Configuration"])


def programming_libraries_table() -> pd.DataFrame:
    distributions = [
        ("NumPy", "numpy"),
        ("pandas", "pandas"),
        ("SciPy", "scipy"),
        ("scikit-learn", "scikit-learn"),
        ("joblib", "joblib"),
        ("XGBoost", "xgboost"),
        ("Matplotlib", "matplotlib"),
        ("SHAP", "shap"),
        ("psutil", "psutil"),
        ("Jupyter", "jupyter"),
        ("IPython kernel", "ipykernel"),
    ]
    return pd.DataFrame(
        [(label, installed_version(distribution)) for label, distribution in distributions],
        columns=["Library", "Version"],
    )


def _timing_summary(
    values: pd.Series,
    stage: str,
    model: str,
    notes: str,
) -> dict[str, Any]:
    numeric = pd.to_numeric(values, errors="coerce").dropna()
    if numeric.empty:
        return {
            "Stage": stage,
            "Model": model,
            "Timed operations": 0,
            "Total seconds": np.nan,
            "Total minutes": np.nan,
            "Mean seconds": np.nan,
            "SD seconds": np.nan,
            "Minimum seconds": np.nan,
            "Maximum seconds": np.nan,
            "Notes": notes,
        }
    return {
        "Stage": stage,
        "Model": model,
        "Timed operations": int(numeric.size),
        "Total seconds": float(numeric.sum()),
        "Total minutes": float(numeric.sum() / 60.0),
        "Mean seconds": float(numeric.mean()),
        "SD seconds": float(numeric.std(ddof=1)) if numeric.size > 1 else 0.0,
        "Minimum seconds": float(numeric.min()),
        "Maximum seconds": float(numeric.max()),
        "Notes": notes,
    }


def computation_time_table(
    output_dir: Path,
    loss_curve_timing: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Aggregate durations without double-counting Development/Test metric rows."""
    records: list[dict[str, Any]] = []

    hyperparameter_path = output_dir / "selected_hyperparameters.csv"
    if hyperparameter_path.is_file():
        tuned = pd.read_csv(hyperparameter_path)
        if "tuning_seconds" in tuned:
            for model, subset in tuned.groupby("model", sort=False):
                records.append(
                    _timing_summary(
                        subset["tuning_seconds"],
                        "Hyperparameter optimization",
                        str(model),
                        "One timed search per target, missing type, and feature configuration.",
                    )
                )

    run_metrics_path = output_dir / "run_metrics_all_models.csv"
    if run_metrics_path.is_file():
        metrics = pd.read_csv(run_metrics_path)
        required = {"split", "model", "fit_seconds"}
        if required.issubset(metrics.columns):
            fits = metrics.loc[
                (metrics["split"] == "Development") & (metrics["model"] != "Ensemble")
            ].copy()
            for model, subset in fits.groupby("model", sort=False):
                records.append(
                    _timing_summary(
                        subset["fit_seconds"],
                        "Repeated model fitting",
                        str(model),
                        "Development rows only; duplicate Test metric rows are excluded.",
                    )
                )

    if loss_curve_timing is not None and not loss_curve_timing.empty:
        for model, subset in loss_curve_timing.groupby("model", sort=False):
            records.append(
                _timing_summary(
                    subset["fit_seconds"],
                    "Multi-run loss-curve diagnostics",
                    str(model),
                    "Borehole-disjoint development/validation split; diagnostic only.",
                )
            )

    wall_clock_path = output_dir / "pipeline_wall_clock_time.csv"
    if wall_clock_path.is_file():
        wall_clock = pd.read_csv(wall_clock_path)
        if {"stage", "elapsed_seconds"}.issubset(wall_clock.columns):
            for _, row in wall_clock.iterrows():
                records.append(
                    _timing_summary(
                        pd.Series([row["elapsed_seconds"]]),
                        str(row["stage"]),
                        "All models",
                        "End-to-end wall-clock time measured with time.perf_counter().",
                    )
                )

    return pd.DataFrame(records)


def latex_escape(value: Any) -> str:
    if pd.isna(value):
        return "--"
    text = f"{value:.3f}" if isinstance(value, (float, np.floating)) else str(value)
    for original, escaped in [
        ("&", r"\&"), ("%", r"\%"), ("$", r"\$"),
        ("#", r"\#"), ("_", r"\_"),
    ]:
        text = text.replace(original, escaped)
    return text


def dataframe_to_latex(frame: pd.DataFrame, caption: str, label: str) -> str:
    alignment = "l" + "r" * max(0, len(frame.columns) - 1)
    header = " & ".join(latex_escape(column) for column in frame.columns) + r" \\"
    rows = [
        " & ".join(latex_escape(value) for value in row) + r" \\"
        for row in frame.itertuples(index=False, name=None)
    ]
    return "\n".join([
        r"\begin{table*}[htbp]",
        r"\centering",
        f"\\caption{{{latex_escape(caption)}}}",
        f"\\label{{{label}}}",
        r"\resizebox{\textwidth}{!}{%",
        f"\\begin{{tabular}}{{{alignment}}}",
        r"\toprule",
        header,
        r"\midrule",
        *rows,
        r"\bottomrule",
        r"\end{tabular}%",
        r"}",
        r"\end{table*}",
    ])


def write_reproducibility_report(
    output_dir: Path,
    loss_curve_timing: pd.DataFrame | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    output_dir.mkdir(parents=True, exist_ok=True)
    system = system_configuration_table()
    libraries = programming_libraries_table()
    timings = computation_time_table(output_dir, loss_curve_timing)

    system.to_csv(output_dir / "system_configuration.csv", index=False)
    libraries.to_csv(output_dir / "programming_libraries.csv", index=False)
    timings.to_csv(output_dir / "computation_times.csv", index=False)

    latex_parts = [
        r"% Requires \usepackage{booktabs} and \usepackage{graphicx}",
        dataframe_to_latex(
            system,
            "System configuration used for the experiments.",
            "tab:system_configuration",
        ),
        dataframe_to_latex(
            libraries,
            "Principal programming libraries and versions.",
            "tab:programming_libraries",
        ),
        dataframe_to_latex(
            timings,
            "Measured computation times for model development and diagnostics.",
            "tab:computation_times",
        ),
    ]
    (output_dir / "reproducibility_tables.tex").write_text(
        "\n\n".join(latex_parts), encoding="utf-8"
    )
    return system, libraries, timings


## Genuine spatial-comparator experiment

This section compares the locked **XYZ equal-weight ensemble** with three genuine spatial approaches under the same Study~C Missing-Type definitions, held-out test observations, targets, and error metrics:

1. **Local ordinary kriging (OK):** predicts each target from standardized (x), (y), and (z_{\mathrm{depth}}) coordinates using a spherical semivariogram and the nearest development observations.
2. **Regression--kriging (RK):** fits a ridge trend to the same non-spatial index/mechanical predictors available in each Missing Type, forms borehole-grouped out-of-fold residuals, and kriges those residuals in XYZ space. The out-of-fold construction prevents in-sample residual shrinkage from making the kriging correction artificially favorable.
3. **Local Gaussian-process regression (GPR):** predicts each target from standardized XYZ coordinates using a Mat\'ern-(3/2) covariance and white-noise term.

Spatial scaling, variogram estimation, regression tuning, and neighborhood selection use development data only. The fixed Study~C test labels are used once for final evaluation. The deterministic spatial comparators are fitted once; the ensemble entry is summarized over its existing 40 seeded refits. Therefore, the resulting table is a descriptive held-out comparison and is not used for paired Wilcoxon inference.


In [15]:
# Spatial-comparator imports and settings
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict


SPATIAL_COORDINATES = ["x", "y", "z_depth"]

# Local neighborhoods make OK and GPR computationally practical while preserving
# genuine distance-based spatial prediction. These values are fixed before the
# held-out test set is evaluated.
OK_NEIGHBORS = 64
GPR_NEIGHBORS = 256
VARIOGRAM_PAIR_SAMPLE = 50_000
VARIOGRAM_RANGE_QUANTILE = 0.75
VARIOGRAM_NUGGET_FRACTION = 0.05
RK_ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]
GPR_WHITE_NOISE = 0.05


def _sample_pair_distances(
    coordinates: np.ndarray,
    max_pairs: int,
    seed: int,
) -> np.ndarray:
    """Sample nonzero pair distances using development coordinates only."""
    coordinates = np.asarray(coordinates, dtype=float)
    count = len(coordinates)
    if count < 2:
        raise ValueError("At least two development observations are required.")
    rng = np.random.default_rng(seed)
    pair_count = min(max_pairs, max(1, count * (count - 1) // 2))
    first = rng.integers(0, count, size=pair_count)
    second = rng.integers(0, count, size=pair_count)
    keep = first != second
    distances = np.linalg.norm(
        coordinates[first[keep]] - coordinates[second[keep]], axis=1
    )
    distances = distances[np.isfinite(distances) & (distances > 0)]
    if distances.size == 0:
        raise ValueError("Development coordinates do not contain nonzero separations.")
    return distances


def estimate_variogram_parameters(
    scaled_coordinates: np.ndarray,
    values: np.ndarray,
    seed: int,
) -> dict[str, float]:
    """Estimate fixed spherical-variogram parameters from development data."""
    distances = _sample_pair_distances(
        scaled_coordinates, VARIOGRAM_PAIR_SAMPLE, seed
    )
    spatial_range = float(np.quantile(distances, VARIOGRAM_RANGE_QUANTILE))
    variance = float(np.var(np.asarray(values, dtype=float), ddof=1))
    variance = max(variance, np.finfo(float).eps)
    nugget = VARIOGRAM_NUGGET_FRACTION * variance
    partial_sill = max(variance - nugget, np.finfo(float).eps)
    return {
        "range": max(spatial_range, np.finfo(float).eps),
        "partial_sill": partial_sill,
        "nugget": nugget,
    }


def spherical_semivariogram(
    distances: np.ndarray,
    spatial_range: float,
    partial_sill: float,
    nugget: float,
) -> np.ndarray:
    distances = np.asarray(distances, dtype=float)
    ratio = np.clip(distances / spatial_range, 0.0, None)
    inside = nugget + partial_sill * (1.5 * ratio - 0.5 * ratio**3)
    semivariance = np.where(
        distances <= spatial_range,
        inside,
        nugget + partial_sill,
    )
    # The semivariance at zero lag is zero; the nugget applies immediately
    # beyond zero and represents microscale variation/measurement noise.
    return np.where(distances == 0.0, 0.0, semivariance)


def local_ordinary_kriging_predict(
    train_coordinates: np.ndarray,
    train_values: np.ndarray,
    query_coordinates: np.ndarray,
    neighbors: int,
    variogram: dict[str, float],
) -> np.ndarray:
    """Local ordinary kriging with an unbiasedness constraint."""
    train_coordinates = np.asarray(train_coordinates, dtype=float)
    train_values = np.asarray(train_values, dtype=float)
    query_coordinates = np.asarray(query_coordinates, dtype=float)
    neighbor_count = min(int(neighbors), len(train_coordinates))
    if neighbor_count < 2:
        raise ValueError("Ordinary kriging requires at least two neighbors.")

    tree = cKDTree(train_coordinates)
    predictions = []
    for query in query_coordinates:
        distances, indices = tree.query(query, k=neighbor_count)
        distances = np.atleast_1d(distances)
        indices = np.atleast_1d(indices)

        exact = np.flatnonzero(distances <= 1e-12)
        if exact.size:
            predictions.append(float(np.mean(train_values[indices[exact]])))
            continue

        local_coordinates = train_coordinates[indices]
        local_values = train_values[indices]
        pairwise = cdist(local_coordinates, local_coordinates)
        gamma_matrix = spherical_semivariogram(
            pairwise,
            variogram["range"],
            variogram["partial_sill"],
            variogram["nugget"],
        )
        gamma_query = spherical_semivariogram(
            cdist(local_coordinates, query.reshape(1, -1)).ravel(),
            variogram["range"],
            variogram["partial_sill"],
            variogram["nugget"],
        )

        system = np.empty((neighbor_count + 1, neighbor_count + 1), dtype=float)
        system[:neighbor_count, :neighbor_count] = gamma_matrix
        system[:neighbor_count, -1] = 1.0
        system[-1, :neighbor_count] = 1.0
        system[-1, -1] = 0.0
        rhs = np.append(gamma_query, 1.0)
        # A very small diagonal term stabilizes duplicate/near-duplicate locations.
        system[:neighbor_count, :neighbor_count] += np.eye(neighbor_count) * 1e-10
        try:
            solution = np.linalg.solve(system, rhs)
        except np.linalg.LinAlgError:
            solution = np.linalg.lstsq(system, rhs, rcond=None)[0]
        predictions.append(float(np.dot(solution[:neighbor_count], local_values)))
    return np.asarray(predictions)


def prepare_spatial_coordinates(
    development: pd.DataFrame,
    evaluation: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, StandardScaler]:
    scaler = StandardScaler()
    development_coordinates = scaler.fit_transform(
        development.loc[:, SPATIAL_COORDINATES]
    )
    evaluation_coordinates = scaler.transform(
        evaluation.loc[:, SPATIAL_COORDINATES]
    )
    return development_coordinates, evaluation_coordinates, scaler


In [16]:
def fit_regression_kriging(
    task: ExperimentTask,
    development: pd.DataFrame,
    evaluation: pd.DataFrame,
    args: argparse.Namespace,
) -> tuple[np.ndarray, dict[str, Any]]:
    """Ridge trend plus ordinary kriging of grouped out-of-fold residuals."""
    non_spatial_features = [
        feature for feature in task.features if feature not in SPATIAL_COORDINATES
    ]
    X_development = development.loc[:, non_spatial_features]
    X_evaluation = evaluation.loc[:, non_spatial_features]
    y_development = development.loc[:, task.target].to_numpy(dtype=float)

    trend = Pipeline(
        [
            (
                "poly",
                PolynomialFeatures(
                    degree=task.polynomial_degree,
                    include_bias=False,
                ),
            ),
            ("scale", StandardScaler()),
            ("ridge", Ridge()),
        ]
    )
    cv, groups = make_cv(
        args.cv_strategy,
        args.cv_folds,
        args.tuning_seed,
        development["borehole_id"],
    )
    search = GridSearchCV(
        trend,
        param_grid={"ridge__alpha": RK_ALPHA_GRID},
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=args.n_jobs,
        refit=True,
    )
    fit_kwargs = {"groups": groups} if groups is not None else {}
    search.fit(X_development, y_development, **fit_kwargs)
    locked_trend = clone(search.best_estimator_)

    prediction_kwargs = {
        "estimator": locked_trend,
        "X": X_development,
        "y": y_development,
        "cv": cv,
        "n_jobs": args.n_jobs,
        "method": "predict",
    }
    if groups is not None:
        prediction_kwargs["groups"] = groups
    out_of_fold_trend = cross_val_predict(**prediction_kwargs)
    residuals = y_development - out_of_fold_trend

    locked_trend.fit(X_development, y_development)
    evaluation_trend = locked_trend.predict(X_evaluation)
    development_coordinates, evaluation_coordinates, _ = prepare_spatial_coordinates(
        development, evaluation
    )
    residual_variogram = estimate_variogram_parameters(
        development_coordinates, residuals, args.tuning_seed
    )
    residual_correction = local_ordinary_kriging_predict(
        development_coordinates,
        residuals,
        evaluation_coordinates,
        neighbors=OK_NEIGHBORS,
        variogram=residual_variogram,
    )
    metadata = {
        "trend_features": ", ".join(non_spatial_features),
        "ridge_alpha": float(search.best_params_["ridge__alpha"]),
        "variogram_range": residual_variogram["range"],
        "variogram_partial_sill": residual_variogram["partial_sill"],
        "variogram_nugget": residual_variogram["nugget"],
        "neighbors": min(OK_NEIGHBORS, len(development)),
    }
    return evaluation_trend + residual_correction, metadata


def fit_spatial_comparators_for_task(
    task: ExperimentTask,
    development: pd.DataFrame,
    evaluation: pd.DataFrame,
    args: argparse.Namespace,
) -> tuple[dict[str, np.ndarray], list[dict[str, Any]]]:
    """Fit OK, RK, and local GPR using development data only."""
    y_development = development.loc[:, task.target].to_numpy(dtype=float)
    development_coordinates, evaluation_coordinates, _ = prepare_spatial_coordinates(
        development, evaluation
    )
    predictions: dict[str, np.ndarray] = {}
    metadata_rows: list[dict[str, Any]] = []

    started = time.perf_counter()
    variogram = estimate_variogram_parameters(
        development_coordinates, y_development, args.tuning_seed
    )
    predictions["Ordinary Kriging"] = local_ordinary_kriging_predict(
        development_coordinates,
        y_development,
        evaluation_coordinates,
        neighbors=OK_NEIGHBORS,
        variogram=variogram,
    )
    metadata_rows.append(
        {
            "method": "Ordinary Kriging",
            "fit_predict_seconds": time.perf_counter() - started,
            "features": ", ".join(SPATIAL_COORDINATES),
            "neighbors": min(OK_NEIGHBORS, len(development)),
            "variogram_range": variogram["range"],
            "variogram_partial_sill": variogram["partial_sill"],
            "variogram_nugget": variogram["nugget"],
            "ridge_alpha": np.nan,
        }
    )

    started = time.perf_counter()
    rk_prediction, rk_metadata = fit_regression_kriging(
        task, development, evaluation, args
    )
    predictions["Regression Kriging"] = rk_prediction
    metadata_rows.append(
        {
            "method": "Regression Kriging",
            "fit_predict_seconds": time.perf_counter() - started,
            "features": rk_metadata["trend_features"] + "; residual XYZ",
            "neighbors": rk_metadata["neighbors"],
            "variogram_range": rk_metadata["variogram_range"],
            "variogram_partial_sill": rk_metadata["variogram_partial_sill"],
            "variogram_nugget": rk_metadata["variogram_nugget"],
            "ridge_alpha": rk_metadata["ridge_alpha"],
        }
    )

    started = time.perf_counter()
    neighbor_count = min(GPR_NEIGHBORS, len(development))
    tree = cKDTree(development_coordinates)
    gpr_predictions = []
    # The Mat\'ern length scale is fixed from the development-coordinate range;
    # optimizer=None prevents the held-out observations from influencing it.
    length_scale = max(variogram["range"] / 2.0, 0.10)
    for query in evaluation_coordinates:
        _, indices = tree.query(query, k=neighbor_count)
        indices = np.atleast_1d(indices)
        kernel = (
            ConstantKernel(1.0, constant_value_bounds="fixed")
            * Matern(
                length_scale=np.repeat(length_scale, 3),
                length_scale_bounds="fixed",
                nu=1.5,
            )
            + WhiteKernel(
                noise_level=GPR_WHITE_NOISE,
                noise_level_bounds="fixed",
            )
        )
        model = GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-8,
            optimizer=None,
            normalize_y=True,
            random_state=args.tuning_seed,
        )
        model.fit(development_coordinates[indices], y_development[indices])
        gpr_predictions.append(float(model.predict(query.reshape(1, -1))[0]))
    predictions["Gaussian Process Regression"] = np.asarray(gpr_predictions)
    metadata_rows.append(
        {
            "method": "Gaussian Process Regression",
            "fit_predict_seconds": time.perf_counter() - started,
            "features": ", ".join(SPATIAL_COORDINATES),
            "neighbors": neighbor_count,
            "variogram_range": np.nan,
            "variogram_partial_sill": np.nan,
            "variogram_nugget": GPR_WHITE_NOISE,
            "ridge_alpha": np.nan,
            "gpr_kernel": f"Matern(nu=1.5, length_scale={length_scale:.6g}) + WhiteKernel",
        }
    )
    return predictions, metadata_rows


In [17]:
def _simple_latex_table(frame: pd.DataFrame) -> str:
    """Render a compact tabular without requiring pandas' optional Jinja2 dependency."""
    def header_text(value: Any) -> str:
        return str(value).replace("_", r"\_").replace("%", r"\%")

    def body_text(value: Any) -> str:
        text = str(value)
        if "$" in text:
            return text
        return text.replace("_", r"\_").replace("%", r"\%")

    alignment = "l" * len(frame.columns)
    lines = [
        rf"\begin{{tabular}}{{{alignment}}}",
        r"\hline",
        " & ".join(header_text(column) for column in frame.columns) + r" \\",
        r"\hline",
    ]
    for row in frame.itertuples(index=False, name=None):
        lines.append(" & ".join(body_text(value) for value in row) + r" \\")
    lines.extend([r"\hline", r"\end{tabular}", ""])
    return "\n".join(lines)


def run_spatial_comparator_experiment(
    notebook_args: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Run all 14 Study C task comparisons and write manuscript-ready outputs."""
    args = notebook_args_namespace(notebook_args)
    output_dir = Path(args.output_dir).expanduser().resolve()
    comparator_dir = output_dir / "spatial_comparators"
    comparator_dir.mkdir(parents=True, exist_ok=True)

    train = load_and_engineer(resolve_input_path(args.train))
    supplied_test = load_and_engineer(resolve_input_path(args.test))
    development, test, _ = prepare_splits(train, supplied_test)

    tasks = [
        task for task in iter_tasks(args)
        if task.configuration == "XYZ"
    ]
    if len(tasks) == 0:
        raise ValueError("No XYZ tasks were selected by NOTEBOOK_ARGS.")

    metric_rows: list[dict[str, Any]] = []
    prediction_rows: list[dict[str, Any]] = []
    metadata_rows: list[dict[str, Any]] = []
    for task_number, task in enumerate(tasks, start=1):
        print(
            f"[{task_number}/{len(tasks)}] Spatial comparators for "
            f"{task.missing_type} {task.target}...",
            flush=True,
        )
        task_test = select_test_rows(test, task.missing_type)
        truth = task_test.loc[:, task.target].to_numpy(dtype=float)
        predictions, method_metadata = fit_spatial_comparators_for_task(
            task, development, task_test, args
        )
        for row in method_metadata:
            row.update(
                {
                    "missing_type": task.missing_type,
                    "target": task.target,
                    "development_records": len(development),
                    "development_boreholes": development["borehole_id"].nunique(),
                    "test_records": len(task_test),
                }
            )
            metadata_rows.append(row)

        for method, predicted in predictions.items():
            values = calculate_metrics(truth, predicted, args.huber_delta)
            metric_rows.append(
                {
                    "missing_type": task.missing_type,
                    "target": task.target,
                    "configuration": "XYZ",
                    "method": method,
                    "n_fits": 1,
                    "n_test": len(truth),
                    **values,
                }
            )
            for position, (_, source_row) in enumerate(task_test.iterrows()):
                prediction_rows.append(
                    {
                        "missing_type": task.missing_type,
                        "target": task.target,
                        "configuration": "XYZ",
                        "method": method,
                        "test_row": (
                    int(source_row["rows"])
                    if "rows" in source_row.index
                    else int(position)
                ),
                        "global_id": source_row["global_id"],
                        "borehole_id": source_row["borehole_id"],
                        "observed": float(truth[position]),
                        "predicted": float(predicted[position]),
                        "residual": float(truth[position] - predicted[position]),
                    }
                )

    comparator_metrics = pd.DataFrame(metric_rows)
    comparator_predictions = pd.DataFrame(prediction_rows)
    comparator_metadata = pd.DataFrame(metadata_rows)

    ensemble_path = output_dir / "run_metrics_all_models.csv"
    if not ensemble_path.is_file():
        raise FileNotFoundError(
            f"{ensemble_path} was not found. Run the complete ensemble experiment first; "
            "the comparator section reuses its locked 40-run XYZ results."
        )
    ensemble_runs = pd.read_csv(ensemble_path)
    ensemble_runs = ensemble_runs.loc[
        (ensemble_runs["configuration"] == "XYZ")
        & (ensemble_runs["model"] == "Ensemble")
        & (ensemble_runs["split"] == "Test")
    ].copy()
    metric_names = ["rmse", "rmse_percent", "l2", "robust_l2", "mae", "r2"]
    ensemble_summary = (
        ensemble_runs.groupby(["missing_type", "target"], as_index=False)[metric_names]
        .agg(["mean", "std"])
    )
    ensemble_summary.columns = [
        "_".join(column).rstrip("_")
        if isinstance(column, tuple) else column
        for column in ensemble_summary.columns
    ]
    ensemble_summary = ensemble_summary.rename(
        columns={
            "missing_type_": "missing_type",
            "target_": "target",
        }
    )
    # pandas versions differ in how they flatten group labels; normalize both forms.
    if "missing_type" not in ensemble_summary.columns:
        ensemble_summary = ensemble_summary.rename(columns={"missing_type_": "missing_type"})
    if "target" not in ensemble_summary.columns:
        ensemble_summary = ensemble_summary.rename(columns={"target_": "target"})
    counts = (
        ensemble_runs.groupby(["missing_type", "target"])
        .size()
        .rename("ensemble_n_fits")
        .reset_index()
    )
    ensemble_summary = ensemble_summary.merge(
        counts, on=["missing_type", "target"], how="left", validate="one_to_one"
    )

    rmse_table = comparator_metrics.pivot(
        index=["missing_type", "target"], columns="method", values="rmse"
    ).reset_index()
    rmse_table = rmse_table.merge(
        ensemble_summary[
            ["missing_type", "target", "rmse_mean", "rmse_std", "ensemble_n_fits"]
        ],
        on=["missing_type", "target"],
        how="left",
        validate="one_to_one",
    )
    comparator_columns = [
        "Ordinary Kriging",
        "Regression Kriging",
        "Gaussian Process Regression",
    ]
    rmse_table["best_spatial_comparator"] = rmse_table[comparator_columns].idxmin(axis=1)
    rmse_table["best_spatial_comparator_rmse"] = rmse_table[comparator_columns].min(axis=1)
    rmse_table["ensemble_vs_best_comparator_improvement_percent"] = (
        100.0
        * (rmse_table["best_spatial_comparator_rmse"] - rmse_table["rmse_mean"])
        / rmse_table["best_spatial_comparator_rmse"]
    )
    rmse_table["best_overall_method"] = rmse_table.apply(
        lambda row: (
            "Ensemble XYZ"
            if row["rmse_mean"] <= row["best_spatial_comparator_rmse"]
            else row["best_spatial_comparator"]
        ),
        axis=1,
    )
    rmse_table = rmse_table.rename(
        columns={
            "rmse_mean": "Ensemble XYZ mean RMSE",
            "rmse_std": "Ensemble XYZ SD RMSE",
        }
    )

    comparator_metrics.to_csv(
        comparator_dir / "spatial_comparator_metrics.csv", index=False
    )
    comparator_predictions.to_csv(
        comparator_dir / "spatial_comparator_predictions.csv", index=False
    )
    comparator_metadata.to_csv(
        comparator_dir / "spatial_comparator_settings_and_timing.csv", index=False
    )
    ensemble_summary.to_csv(
        comparator_dir / "ensemble_xyz_40run_metric_summary.csv", index=False
    )
    rmse_table.to_csv(
        comparator_dir / "ensemble_vs_spatial_comparators_rmse.csv", index=False
    )

    manuscript_table = rmse_table.copy()
    manuscript_table["Ensemble XYZ"] = manuscript_table.apply(
        lambda row: (
            f'{row["Ensemble XYZ mean RMSE"]:.4f} '
            f'$\\pm$ {row["Ensemble XYZ SD RMSE"]:.4f}'
        ),
        axis=1,
    )
    manuscript_table = manuscript_table[
        [
            "missing_type",
            "target",
            "Ensemble XYZ",
            "Ordinary Kriging",
            "Regression Kriging",
            "Gaussian Process Regression",
            "best_overall_method",
        ]
    ].rename(
        columns={
            "missing_type": "Missing Type",
            "target": "Target",
            "best_overall_method": "Lowest RMSE",
        }
    )
    for column in [
        "Ordinary Kriging",
        "Regression Kriging",
        "Gaussian Process Regression",
    ]:
        manuscript_table[column] = manuscript_table[column].map(lambda value: f"{value:.4f}")
    (comparator_dir / "ensemble_vs_spatial_comparators_rmse.tex").write_text(
        _simple_latex_table(manuscript_table), encoding="utf-8"
    )

    ensemble_wins = int((rmse_table["best_overall_method"] == "Ensemble XYZ").sum())
    print(
        f"Ensemble XYZ has the lowest mean RMSE in {ensemble_wins} of "
        f"{len(rmse_table)} task comparisons."
    )
    print(f"Comparator outputs written to {comparator_dir}")
    display(manuscript_table)
    return comparator_metrics, comparator_predictions, rmse_table, comparator_metadata


### Run the spatial comparison

First run the complete ensemble experiment so that `run_metrics_all_models.csv` contains the locked 40-run XYZ ensemble results. Then set `RUN_SPATIAL_COMPARATORS = True` below and execute the cell. The outputs include observation-level predictions, all six metrics, method settings/timings, a direct RMSE comparison, and a LaTeX table.


In [18]:
RUN_SPATIAL_COMPARATORS = False

spatial_comparator_metrics = pd.DataFrame()
spatial_comparator_predictions = pd.DataFrame()
ensemble_spatial_rmse_comparison = pd.DataFrame()
spatial_comparator_metadata = pd.DataFrame()

if RUN_SPATIAL_COMPARATORS:
    (
        spatial_comparator_metrics,
        spatial_comparator_predictions,
        ensemble_spatial_rmse_comparison,
        spatial_comparator_metadata,
    ) = run_spatial_comparator_experiment(NOTEBOOK_ARGS)
else:
    print(
        "Spatial comparator functions are ready. Run the complete ensemble experiment, "
        "then set RUN_SPATIAL_COMPARATORS = True and rerun this cell."
    )


Spatial comparator functions are ready. Run the complete ensemble experiment, then set RUN_SPATIAL_COMPARATORS = True and rerun this cell.


### Spatial-comparator outputs

- `spatial_comparators/spatial_comparator_metrics.csv`: held-out RMSE, RMSE%, L2, Robust L2, MAE, and (R^2) for OK, RK, and GPR.
- `spatial_comparators/spatial_comparator_predictions.csv`: observed and predicted values for every evaluated test observation.
- `spatial_comparators/spatial_comparator_settings_and_timing.csv`: neighborhoods, fitted development-only variogram parameters, selected RK ridge penalty, and timing.
- `spatial_comparators/ensemble_xyz_40run_metric_summary.csv`: mean and SD of the locked 40-run XYZ ensemble results.
- `spatial_comparators/ensemble_vs_spatial_comparators_rmse.csv`: direct task-wise comparison and the percentage difference between the ensemble and best spatial comparator.
- `spatial_comparators/ensemble_vs_spatial_comparators_rmse.tex`: manuscript-ready comparison table.

For reporting, state explicitly that the ensemble values are mean (\pm) SD across 40 stochastic refits, whereas each deterministic spatial comparator is a single locked fit on the same development and test cases. Do not include the comparator table in the existing paired Wilcoxon analysis unless the spatial comparators are also embedded in a defensible resampling design.


## Matched 40-run spatial-comparator analysis

Merely changing a random seed does not create meaningful variation for ordinary kriging, regression--kriging with fixed settings, or fixed-kernel Gaussian-process regression because these methods are deterministic. This section therefore uses 40 matched borehole-level development resamples. For each run, the same reproducibly selected subset of development boreholes is supplied to the XYZ ensemble, local ordinary kriging, regression--kriging, and local GPR. The held-out Study~C test observations remain unchanged.

The ensemble hyperparameters and the regression--kriging ridge penalty are selected from development data and then locked before the repeated comparison. Within every run, regression--kriging constructs residuals using borehole-grouped out-of-fold predictions. The resulting mean and sample standard deviation quantify sensitivity to the composition of the development boreholes. They are not predictive intervals and do not represent uncertainty from independently resampled test observations or external sites.


In [19]:
# Settings for the matched repeated comparison
MATCHED_COMPARISON_RUNS = 40
MATCHED_DEVELOPMENT_BOREHOLE_FRACTION = 0.80
MATCHED_GPR_NEIGHBORS = 96


def matched_borehole_subsamples(
    development: pd.DataFrame,
    runs: int,
    fraction: float,
    base_seed: int,
) -> tuple[list[pd.DataFrame], pd.DataFrame]:
    """Create reproducible run-specific subsets while retaining whole boreholes."""
    if runs < 2:
        raise ValueError("At least two runs are required to estimate a sample standard deviation.")
    if not 0.0 < fraction < 1.0:
        raise ValueError("The development borehole fraction must be between 0 and 1.")
    boreholes = np.asarray(sorted(development["borehole_id"].unique(), key=str))
    selected_count = max(2, int(round(fraction * len(boreholes))))
    if selected_count >= len(boreholes):
        raise ValueError(
            "The selected fraction retains every borehole and would not create "
            "development-sample variability. Use a fraction smaller than 1.0."
        )

    subsets: list[pd.DataFrame] = []
    manifest_rows: list[dict[str, Any]] = []
    for run_index in range(runs):
        seed = base_seed + run_index
        rng = np.random.default_rng(seed)
        selected_boreholes = rng.choice(
            boreholes, size=selected_count, replace=False
        )
        selected_set = set(selected_boreholes.tolist())
        subset = development.loc[
            development["borehole_id"].isin(selected_set)
        ].copy()
        subset = subset.sort_values("global_id").reset_index(drop=True)
        if subset["borehole_id"].nunique() != selected_count:
            raise RuntimeError("The borehole-level resample was not constructed correctly.")
        subsets.append(subset)
        manifest_rows.append(
            {
                "run": run_index + 1,
                "seed": seed,
                "development_fraction_requested": fraction,
                "selected_boreholes": selected_count,
                "available_boreholes": len(boreholes),
                "selected_records": len(subset),
                "borehole_ids": ", ".join(map(str, selected_boreholes.tolist())),
            }
        )
    return subsets, pd.DataFrame(manifest_rows)


def select_locked_rk_alpha(
    task: ExperimentTask,
    development: pd.DataFrame,
    args: argparse.Namespace,
) -> float:
    """Select the RK trend penalty once using development-only grouped CV."""
    non_spatial_features = [
        feature for feature in task.features if feature not in SPATIAL_COORDINATES
    ]
    trend = Pipeline(
        [
            (
                "poly",
                PolynomialFeatures(
                    degree=task.polynomial_degree,
                    include_bias=False,
                ),
            ),
            ("scale", StandardScaler()),
            ("ridge", Ridge()),
        ]
    )
    cv, groups = make_cv(
        args.cv_strategy,
        args.cv_folds,
        args.tuning_seed,
        development["borehole_id"],
    )
    search = GridSearchCV(
        trend,
        param_grid={"ridge__alpha": RK_ALPHA_GRID},
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=args.n_jobs,
        refit=True,
    )
    fit_kwargs = {"groups": groups} if groups is not None else {}
    search.fit(
        development.loc[:, non_spatial_features],
        development.loc[:, task.target],
        **fit_kwargs,
    )
    return float(search.best_params_["ridge__alpha"])


def regression_kriging_predict_locked(
    task: ExperimentTask,
    development_subset: pd.DataFrame,
    evaluation: pd.DataFrame,
    args: argparse.Namespace,
    ridge_alpha: float,
) -> np.ndarray:
    """Fit the locked ridge trend and krige grouped out-of-fold residuals."""
    non_spatial_features = [
        feature for feature in task.features if feature not in SPATIAL_COORDINATES
    ]
    X_development = development_subset.loc[:, non_spatial_features]
    X_evaluation = evaluation.loc[:, non_spatial_features]
    y_development = development_subset.loc[:, task.target].to_numpy(dtype=float)
    trend = Pipeline(
        [
            (
                "poly",
                PolynomialFeatures(
                    degree=task.polynomial_degree,
                    include_bias=False,
                ),
            ),
            ("scale", StandardScaler()),
            ("ridge", Ridge(alpha=ridge_alpha)),
        ]
    )
    cv, groups = make_cv(
        args.cv_strategy,
        args.cv_folds,
        args.tuning_seed,
        development_subset["borehole_id"],
    )
    prediction_kwargs = {
        "estimator": clone(trend),
        "X": X_development,
        "y": y_development,
        "cv": cv,
        "n_jobs": args.n_jobs,
        "method": "predict",
    }
    if groups is not None:
        prediction_kwargs["groups"] = groups
    out_of_fold_prediction = cross_val_predict(**prediction_kwargs)
    residuals = y_development - out_of_fold_prediction
    trend.fit(X_development, y_development)
    trend_prediction = trend.predict(X_evaluation)

    development_coordinates, evaluation_coordinates, _ = prepare_spatial_coordinates(
        development_subset, evaluation
    )
    residual_variogram = estimate_variogram_parameters(
        development_coordinates, residuals, args.tuning_seed
    )
    residual_prediction = local_ordinary_kriging_predict(
        development_coordinates,
        residuals,
        evaluation_coordinates,
        neighbors=OK_NEIGHBORS,
        variogram=residual_variogram,
    )
    return trend_prediction + residual_prediction


In [20]:
def locked_ensemble_predict(
    task: ExperimentTask,
    development_subset: pd.DataFrame,
    evaluation: pd.DataFrame,
    selected_hyperparameters: pd.DataFrame,
    args: argparse.Namespace,
    seed: int,
) -> np.ndarray:
    """Refit the locked RF, GBR, and XGB models on one matched resample."""
    predictions = []
    X_development = development_subset.loc[:, task.features]
    y_development = development_subset.loc[:, task.target]
    X_evaluation = evaluation.loc[:, task.features]
    for model_name in ["RF", "GBR", "XGB"]:
        model_parameters = selected_model_parameters(
            selected_hyperparameters,
            task.missing_type,
            task.configuration,
            task.target,
            model_name,
        )
        estimator = build_estimator(
            model_name,
            task.polynomial_degree,
            seed,
        )
        estimator.set_params(
            **{
                f"regressor__model__{name}": value
                for name, value in model_parameters.items()
            }
        )
        estimator = reseed_estimator(estimator, seed, args.n_jobs)
        estimator.fit(X_development, y_development)
        predictions.append(estimator.predict(X_evaluation))
    return np.mean(np.column_stack(predictions), axis=1)


def ordinary_kriging_predict_for_run(
    task: ExperimentTask,
    development_subset: pd.DataFrame,
    evaluation: pd.DataFrame,
    seed: int,
) -> np.ndarray:
    y_development = development_subset.loc[:, task.target].to_numpy(dtype=float)
    development_coordinates, evaluation_coordinates, _ = prepare_spatial_coordinates(
        development_subset, evaluation
    )
    variogram = estimate_variogram_parameters(
        development_coordinates, y_development, seed
    )
    return local_ordinary_kriging_predict(
        development_coordinates,
        y_development,
        evaluation_coordinates,
        neighbors=OK_NEIGHBORS,
        variogram=variogram,
    )


def local_gpr_predict_for_run(
    task: ExperimentTask,
    development_subset: pd.DataFrame,
    evaluation: pd.DataFrame,
    seed: int,
) -> np.ndarray:
    y_development = development_subset.loc[:, task.target].to_numpy(dtype=float)
    development_coordinates, evaluation_coordinates, _ = prepare_spatial_coordinates(
        development_subset, evaluation
    )
    variogram = estimate_variogram_parameters(
        development_coordinates, y_development, seed
    )
    length_scale = max(variogram["range"] / 2.0, 0.10)
    neighbor_count = min(MATCHED_GPR_NEIGHBORS, len(development_subset))
    tree = cKDTree(development_coordinates)
    predictions = []
    for query in evaluation_coordinates:
        _, indices = tree.query(query, k=neighbor_count)
        indices = np.atleast_1d(indices)
        kernel = (
            ConstantKernel(1.0, constant_value_bounds="fixed")
            * Matern(
                length_scale=np.repeat(length_scale, 3),
                length_scale_bounds="fixed",
                nu=1.5,
            )
            + WhiteKernel(
                noise_level=GPR_WHITE_NOISE,
                noise_level_bounds="fixed",
            )
        )
        model = GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-8,
            optimizer=None,
            normalize_y=True,
            random_state=seed,
        )
        model.fit(development_coordinates[indices], y_development[indices])
        predictions.append(float(model.predict(query.reshape(1, -1))[0]))
    return np.asarray(predictions)


In [21]:
def summarize_matched_comparison(run_metrics: pd.DataFrame) -> pd.DataFrame:
    metric_names = ["rmse", "rmse_percent", "l2", "robust_l2", "mae", "r2"]
    rows: list[dict[str, Any]] = []
    for keys, group in run_metrics.groupby(
        ["missing_type", "target", "method"], sort=False
    ):
        missing_type, target, method = keys
        record: dict[str, Any] = {
            "missing_type": missing_type,
            "target": target,
            "method": method,
            "runs": len(group),
            "test_records_per_run": int(group["n_test"].iloc[0]),
        }
        for metric in metric_names:
            values = pd.to_numeric(group[metric], errors="coerce")
            record[f"{metric}_mean"] = float(values.mean())
            record[f"{metric}_sd"] = float(values.std(ddof=1))
        rows.append(record)
    return pd.DataFrame(rows)


def matched_rmse_table(summary: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    method_order = [
        "Ensemble XYZ",
        "Ordinary Kriging",
        "Regression Kriging",
        "Gaussian Process Regression",
    ]
    numerical_rows: list[dict[str, Any]] = []
    formatted_rows: list[dict[str, Any]] = []
    for (missing_type, target), group in summary.groupby(
        ["missing_type", "target"], sort=False
    ):
        numerical = {"Missing Type": missing_type, "Target": target}
        formatted = {"Missing Type": missing_type, "Target": target}
        means: dict[str, float] = {}
        for method in method_order:
            match = group.loc[group["method"] == method]
            if len(match) != 1:
                raise ValueError(
                    f"Expected one summary row for {missing_type}/{target}/{method}."
                )
            mean_value = float(match.iloc[0]["rmse_mean"])
            sd_value = float(match.iloc[0]["rmse_sd"])
            means[method] = mean_value
            numerical[f"{method} mean RMSE"] = mean_value
            numerical[f"{method} SD RMSE"] = sd_value
            formatted[method] = f"{mean_value:.4f} $\\pm$ {sd_value:.4f}"
        best_method = min(means, key=means.get)
        numerical["Lowest mean RMSE"] = best_method
        formatted["Lowest mean RMSE"] = best_method
        numerical_rows.append(numerical)
        formatted_rows.append(formatted)
    return pd.DataFrame(numerical_rows), pd.DataFrame(formatted_rows)


def paired_ensemble_comparator_differences(
    run_metrics: pd.DataFrame,
) -> pd.DataFrame:
    """Describe matched RMSE differences without treating them as test-sample inference."""
    ensemble = run_metrics.loc[
        run_metrics["method"] == "Ensemble XYZ",
        ["run", "seed", "missing_type", "target", "rmse"],
    ].rename(columns={"rmse": "ensemble_rmse"})
    rows: list[dict[str, Any]] = []
    for method in [
        "Ordinary Kriging",
        "Regression Kriging",
        "Gaussian Process Regression",
    ]:
        comparator = run_metrics.loc[
            run_metrics["method"] == method,
            ["run", "seed", "missing_type", "target", "rmse"],
        ].rename(columns={"rmse": "comparator_rmse"})
        paired = ensemble.merge(
            comparator,
            on=["run", "seed", "missing_type", "target"],
            how="inner",
            validate="one_to_one",
        )
        paired["difference_comparator_minus_ensemble"] = (
            paired["comparator_rmse"] - paired["ensemble_rmse"]
        )
        for (missing_type, target), group in paired.groupby(
            ["missing_type", "target"], sort=False
        ):
            differences = group["difference_comparator_minus_ensemble"]
            rows.append(
                {
                    "missing_type": missing_type,
                    "target": target,
                    "comparator": method,
                    "paired_runs": len(group),
                    "mean_rmse_difference_comparator_minus_ensemble": float(
                        differences.mean()
                    ),
                    "sd_rmse_difference": float(differences.std(ddof=1)),
                    "median_rmse_difference": float(differences.median()),
                    "ensemble_lower_rmse_runs": int((differences > 0).sum()),
                }
            )
    return pd.DataFrame(rows)


In [22]:
def run_matched_40run_spatial_comparison(
    notebook_args: list[str],
    runs: int = MATCHED_COMPARISON_RUNS,
    development_fraction: float = MATCHED_DEVELOPMENT_BOREHOLE_FRACTION,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    args = notebook_args_namespace(notebook_args)
    output_dir = Path(args.output_dir).expanduser().resolve()
    comparison_dir = output_dir / "matched_40run_spatial_comparison"
    comparison_dir.mkdir(parents=True, exist_ok=True)

    selected_path = output_dir / "selected_hyperparameters.csv"
    if not selected_path.is_file():
        raise FileNotFoundError(
            f"{selected_path} was not found. Run the complete ensemble experiment "
            "first so this analysis can reuse locked XYZ hyperparameters."
        )
    selected_hyperparameters = pd.read_csv(selected_path)

    train = load_and_engineer(resolve_input_path(args.train))
    supplied_test = load_and_engineer(resolve_input_path(args.test))
    development, test, _ = prepare_splits(train, supplied_test)
    tasks = [task for task in iter_tasks(args) if task.configuration == "XYZ"]
    if len(tasks) != 14 and args.targets is None:
        warnings.warn(
            f"Expected 14 XYZ tasks for the complete study, but found {len(tasks)}."
        )

    print("Selecting and locking the regression-kriging ridge penalties...")
    rk_alphas = {
        (task.missing_type, task.target): select_locked_rk_alpha(
            task, development, args
        )
        for task in tasks
    }
    pd.DataFrame(
        [
            {
                "missing_type": missing_type,
                "target": target,
                "ridge_alpha": alpha,
            }
            for (missing_type, target), alpha in rk_alphas.items()
        ]
    ).to_csv(comparison_dir / "locked_regression_kriging_alphas.csv", index=False)

    subsets, resample_manifest = matched_borehole_subsamples(
        development,
        runs=runs,
        fraction=development_fraction,
        base_seed=args.base_seed,
    )
    metric_rows: list[dict[str, Any]] = []
    prediction_rows: list[dict[str, Any]] = []
    method_order = [
        "Ensemble XYZ",
        "Ordinary Kriging",
        "Regression Kriging",
        "Gaussian Process Regression",
    ]

    for run_index, development_subset in enumerate(subsets):
        run = run_index + 1
        seed = args.base_seed + run_index
        print(f"Matched run {run}/{runs}, seed {seed}...", flush=True)
        for task in tasks:
            task_test = select_test_rows(test, task.missing_type)
            truth = task_test.loc[:, task.target].to_numpy(dtype=float)
            predictions = {
                "Ensemble XYZ": locked_ensemble_predict(
                    task,
                    development_subset,
                    task_test,
                    selected_hyperparameters,
                    args,
                    seed,
                ),
                "Ordinary Kriging": ordinary_kriging_predict_for_run(
                    task, development_subset, task_test, seed
                ),
                "Regression Kriging": regression_kriging_predict_locked(
                    task,
                    development_subset,
                    task_test,
                    args,
                    rk_alphas[(task.missing_type, task.target)],
                ),
                "Gaussian Process Regression": local_gpr_predict_for_run(
                    task, development_subset, task_test, seed
                ),
            }
            for method in method_order:
                predicted = predictions[method]
                values = calculate_metrics(truth, predicted, args.huber_delta)
                metric_rows.append(
                    {
                        "run": run,
                        "seed": seed,
                        "missing_type": task.missing_type,
                        "target": task.target,
                        "configuration": "XYZ",
                        "method": method,
                        "development_records": len(development_subset),
                        "development_boreholes": development_subset[
                            "borehole_id"
                        ].nunique(),
                        "n_test": len(truth),
                        **values,
                    }
                )
                for position, (_, source_row) in enumerate(task_test.iterrows()):
                    prediction_rows.append(
                        {
                            "run": run,
                            "seed": seed,
                            "missing_type": task.missing_type,
                            "target": task.target,
                            "configuration": "XYZ",
                            "method": method,
                            "test_row": (
                    int(source_row["rows"])
                    if "rows" in source_row.index
                    else int(position)
                ),
                            "global_id": source_row["global_id"],
                            "borehole_id": source_row["borehole_id"],
                            "observed": float(truth[position]),
                            "predicted": float(predicted[position]),
                            "residual": float(truth[position] - predicted[position]),
                        }
                    )

    run_metrics = pd.DataFrame(metric_rows)
    predictions = pd.DataFrame(prediction_rows)
    summary = summarize_matched_comparison(run_metrics)
    numerical_rmse, formatted_rmse = matched_rmse_table(summary)
    paired_differences = paired_ensemble_comparator_differences(run_metrics)

    run_metrics.to_csv(comparison_dir / "matched_40run_metrics.csv", index=False)
    predictions.to_csv(comparison_dir / "matched_40run_predictions.csv", index=False)
    summary.to_csv(
        comparison_dir / "matched_40run_summary_mean_sd.csv", index=False
    )
    numerical_rmse.to_csv(
        comparison_dir / "matched_40run_rmse_mean_sd_numeric.csv", index=False
    )
    paired_differences.to_csv(
        comparison_dir / "matched_40run_paired_rmse_differences.csv", index=False
    )
    resample_manifest.to_csv(
        comparison_dir / "matched_40run_borehole_resample_manifest.csv", index=False
    )
    (comparison_dir / "matched_40run_rmse_mean_sd.tex").write_text(
        _simple_latex_table(formatted_rmse), encoding="utf-8"
    )

    print(f"Matched repeated-comparison outputs written to {comparison_dir}")
    display(formatted_rmse)
    return run_metrics, summary, numerical_rmse, paired_differences


### Execute the matched repeated comparison

Run the original complete experiment first so that `selected_hyperparameters.csv` contains the locked XYZ settings. Then set `RUN_MATCHED_40RUN_SPATIAL_COMPARISON = True`. For a short computational check, temporarily set `MATCHED_COMPARISON_RUNS` to 2 or 3; restore it to 40 for the manuscript results.


In [23]:
RUN_MATCHED_40RUN_SPATIAL_COMPARISON = True

matched_run_metrics = pd.DataFrame()
matched_summary = pd.DataFrame()
matched_rmse_comparison = pd.DataFrame()
matched_paired_differences = pd.DataFrame()

if RUN_MATCHED_40RUN_SPATIAL_COMPARISON:
    (
        matched_run_metrics,
        matched_summary,
        matched_rmse_comparison,
        matched_paired_differences,
    ) = run_matched_40run_spatial_comparison(
        NOTEBOOK_ARGS,
        runs=MATCHED_COMPARISON_RUNS,
        development_fraction=MATCHED_DEVELOPMENT_BOREHOLE_FRACTION,
    )
else:
    print(
        "Matched 40-run comparison is ready. Run the complete ensemble analysis, "
        "then set RUN_MATCHED_40RUN_SPATIAL_COMPARISON = True."
    )


Selecting and locking the regression-kriging ridge penalties...
Matched run 1/40, seed 2026...
Matched run 2/40, seed 2027...
Matched run 3/40, seed 2028...
Matched run 4/40, seed 2029...
Matched run 5/40, seed 2030...
Matched run 6/40, seed 2031...
Matched run 7/40, seed 2032...
Matched run 8/40, seed 2033...
Matched run 9/40, seed 2034...
Matched run 10/40, seed 2035...
Matched run 11/40, seed 2036...
Matched run 12/40, seed 2037...
Matched run 13/40, seed 2038...
Matched run 14/40, seed 2039...
Matched run 15/40, seed 2040...
Matched run 16/40, seed 2041...
Matched run 17/40, seed 2042...
Matched run 18/40, seed 2043...
Matched run 19/40, seed 2044...
Matched run 20/40, seed 2045...
Matched run 21/40, seed 2046...
Matched run 22/40, seed 2047...
Matched run 23/40, seed 2048...
Matched run 24/40, seed 2049...
Matched run 25/40, seed 2050...
Matched run 26/40, seed 2051...
Matched run 27/40, seed 2052...
Matched run 28/40, seed 2053...
Matched run 29/40, seed 2054...
Matched run 30/40

,Missing Type,Target,Ensemble XYZ,Ordinary Kriging,Regression Kriging,Gaussian Process Regression,Lowest mean RMSE
0,MT1,Cc,0.1420 $\pm$ 0.0009,0.2729 $\pm$ 0.0051,0.1580 $\pm$ 0.0025,0.2697 $\pm$ 0.0060,Ensemble XYZ
1,MT1,Cv,309.8024 $\pm$ 3.4522,399.9537 $\pm$ 13.2055,355.8350 $\pm$ 13.3522,401.8249 $\pm$ 18.1845,Ensemble XYZ
2,MT1,Pc,71.3731 $\pm$ 1.6014,87.7110 $\pm$ 2.8176,88.7691 $\pm$ 3.4106,88.0158 $\pm$ 2.9784,Ensemble XYZ
3,MT1,E50,2966.5720 $\pm$ 62.0067,209122471040846987264.0000 $\pm$ 3980001589072...,536584293205469691904.0000 $\pm$ 1233517313420...,3075.6601 $\pm$ 78.1848,Ensemble XYZ
4,MT1,su,15.4058 $\pm$ 0.5098,17.4531 $\pm$ 0.6559,18.7212 $\pm$ 0.6802,17.8686 $\pm$ 0.7642,Ensemble XYZ
5,MT2,Cc,0.1411 $\pm$ 0.0008,0.2729 $\pm$ 0.0051,0.1562 $\pm$ 0.0026,0.2697 $\pm$ 0.0060,Ensemble XYZ
6,MT2,Cv,311.1371 $\pm$ 4.5420,399.9537 $\pm$ 13.2055,344.7354 $\pm$ 13.2098,401.8249 $\pm$ 18.1845,Ensemble XYZ
7,MT2,Pc,65.6655 $\pm$ 0.8469,87.7110 $\pm$ 2.8176,82.1989 $\pm$ 3.8248,88.0158 $\pm$ 2.9784,Ensemble XYZ
8,MT2,E50,2384.5521 $\pm$ 37.8570,209122471040846987264.0000 $\pm$ 3980001589072...,9396399415528296448.0000 $\pm$ 430519991732107...,3075.6601 $\pm$ 78.1848,Ensemble XYZ
9,MT3,Cc,0.1419 $\pm$ 0.0010,0.2729 $\pm$ 0.0051,0.1564 $\pm$ 0.0027,0.2697 $\pm$ 0.0060,Ensemble XYZ


### Matched repeated-comparison outputs

- `matched_40run_metrics.csv`: run-level values of all six metrics for the ensemble, OK, RK, and GPR.
- `matched_40run_predictions.csv`: observation-level predictions for all four methods and 40 matched resamples.
- `matched_40run_summary_mean_sd.csv`: mean and sample SD of RMSE, RMSE%, L2, Robust L2, MAE, and (R^2).
- `matched_40run_rmse_mean_sd_numeric.csv`: numerical RMSE mean and SD comparison.
- `matched_40run_rmse_mean_sd.tex`: manuscript-ready mean (\pm) SD RMSE table.
- `matched_40run_paired_rmse_differences.csv`: descriptive paired RMSE differences for identical borehole resamples.
- `matched_40run_borehole_resample_manifest.csv`: the selected boreholes and record count for every run.
- `locked_regression_kriging_alphas.csv`: development-selected RK trend penalties.

The values from this section should replace, rather than be combined with, the original full-development 40-seed ensemble summary when making a direct mean (\pm) SD comparison with the spatial methods. This is necessary because all four methods must share the same source of run-to-run variation.
